# VIDSGEN Phase 6B: backend + ComfyUI on one Kaggle GPU (localhost only)

Runs the **real VIDSGEN FastAPI backend** and **ComfyUI** together on this Kaggle machine and drives the backend through its public API
with one real text-to-video job. Both services listen on `127.0.0.1` only. **No tunnel, no proxy, nothing is exposed.**

Success is never inferred from "the server started" or "the job was queued". The test passes only if **every** check passes:
health, the generation limits rejecting out-of-range requests, a real job with live progress, a completed job, download and
HTTP Range support, an MP4 that fully decodes with the right frames/size/codec and is not blank, and `inference` turning `ok`
in the health report after a real generation.

| Stage | What happens |
|---|---|
| 1-5 | Exactly the verified Phase 5B stages: preflight, pinned ComfyUI install (Kaggle's PyTorch is never replaced), verified model download (16.9 GiB), loopback ComfyUI on GPU 0, live template check |
| 6 Backend install | The backend's own dependencies go into the same venv under the same protection: PyTorch, torchvision, torchaudio and NumPy stay the host's copies and the host's Python environment is verified unchanged |
| 7 End-to-end | The backend starts with `GENERATION_PROFILE=t4_verified` and **one job at a time**; out-of-limit requests are rejected; one job (**832x480, 17 frames, 20 steps, 24 fps, CFG 5.0**) is submitted, tracked, downloaded and verified |

**Before you run it**
1. *Settings -> Accelerator:* **GPU T4 x2** (only GPU 0 is used; VRAM is not pooled). *Internet:* **On**.
2. Optional: *Add-ons -> Secrets* -> `HF_TOKEN` (not needed for these public repositories; never printed or saved).
3. **Run All.** This notebook is self-contained: a new Kaggle session starts empty, so it installs ComfyUI and downloads the models again
   (roughly 15-30 minutes in total). The VIDSGEN repository is public, so no GitHub credential is needed.

**It does NOT:** open a tunnel, expose either service, use `/kaggle/working` for weights, or change your GitHub repository.

In [ ]:
# --- 1. Settings (Phase 6B) (the only cell you should need to edit) ---------------------------------------------------
import json, os, re, subprocess, sys, time
from pathlib import Path

BASE = "/kaggle/tmp/vidsgen-gpu"   # ComfyUI, venv, model weights, caches and temp files. Never /kaggle/working.
GPU = "0"                          # ComfyUI uses this ONE GPU (T4 x2 VRAM is not pooled)
PORT = 8188                        # ComfyUI listens on 127.0.0.1 only

# Pinned sources: nothing here follows a moving branch.
VIDSGEN_REPO, VIDSGEN_COMMIT = "https://github.com/Prosperotechit/VIDSGEN.git", "1f9d5f40fc62a4bb933333d04ab37c7a5de57342"
COMFYUI_REPO = "https://github.com/comfyanonymous/ComfyUI.git"
COMFYUI_COMMIT, COMFYUI_VERSION = "995456d0188ffb51032c28f41527cfa82d87d8a9", "0.46.0"

DOWNLOAD_MODELS = True    # ~16.9 GiB (3 files), verified by exact size + SHA-256. False = stop after preflight + install.
RUN_SMOKE_TEST = True
SMOKE = dict(resolution="832x480", frames=17, steps=20, fps=24, cfg=5.0, seed=424242)
FALLBACK_ARGS = ["--disable-dynamic-vram", "--disable-smart-memory"]   # ONE retry, only after an out-of-memory failure
STOP_SERVER_AT_END = False                                              # leave ComfyUI running for the next phase

OUT = Path("/kaggle/working")      # only the small JSON reports and the test MP4 go here
TOOLS, DATA = f"{BASE}/tools", f"{BASE}/comfy-data"
VENV_PY = f"{BASE}/venv/bin/python"
URL = f"http://127.0.0.1:{PORT}"
STAGES = {}                        # collected for the final report


def run(cmd, env=None, check=True):
    """Run a command and stream its output live. Returns (exit code, output text)."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"STOP: exit code {proc.returncode}. Read the output above, fix the cause, then re-run from this cell.")
    return proc.returncode, "".join(lines)


def last_json(text):
    """The last top-level JSON object printed by one of the tools."""
    start = text.rfind("\n{\n")
    start = 0 if text.startswith("{\n") and start < 0 else start + 1
    return json.loads(text[start:])


def hf_env():
    """Environment for the preflight/downloader. HF_TOKEN comes only from a Kaggle Secret and is never printed."""
    env = dict(os.environ)
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            env["HF_TOKEN"] = token
            print("HF_TOKEN secret: attached (value not shown)")
    except Exception:
        print("HF_TOKEN secret: not attached (fine for these public repositories)")
    return env


def stage_failed(name, why):
    STAGES[name] = {"ok": False, "why": why}
    raise RuntimeError(f"STOP at stage '{name}': {why}")

In [ ]:
# --- 1b. Phase 6B settings -----------------------------------------------------------------------------------------
BACKEND_PORT = 8000          # the FastAPI backend listens on 127.0.0.1 only
RUN_E2E = True
RUN_CANCEL_TEST = False      # True: also cancel a SECOND job while it is being processed (extra GPU time, real interrupt)
E2E_TIMEOUT_S = 2400
STOP_SERVER_AT_END = True    # stop ComfyUI at the end of the notebook (the backend stops itself)
# The test always uses the verified values: 832x480, 17 frames, 20 steps, 24 fps, CFG 5.0, GENERATION_PROFILE=t4_verified, 1 job at a time.

In [ ]:
# --- 2. Tools used by this notebook (embedded verbatim from the VIDSGEN repository) -------------------------
# They are written to /kaggle/tmp; the next three cells contain the exact source, so you can read what will run.
os.makedirs(TOOLS, exist_ok=True)
print("tools directory:", TOOLS)

In [ ]:
# Embedded verbatim from scripts/download_models.py, so this notebook is self-contained. A test keeps this copy identical.
Path(TOOLS, 'download_models.py').write_text(r'''#!/usr/bin/env python3
"""Download the three Wan 2.2 TI2V-5B model files for ComfyUI, from workflows/comfyui/models.yaml.

SAFE BY DEFAULT: without --execute this is a dry run. It only prints which files would be downloaded,
their sizes, pinned revisions and destinations. Nothing is downloaded and no network connection is made.

    python scripts/download_models.py --comfyui-dir DIR                 # dry run (default)
    python scripts/download_models.py --comfyui-dir DIR --execute       # download + verify (~16.9 GiB)
    python scripts/download_models.py --comfyui-dir DIR --verify-only   # hash existing files, no network

What --execute guarantees, for every file:
  * it downloads ONLY the three files listed in the manifest, each from its pinned repository revision
    (an immutable commit), never from a moving branch;
  * it refuses to start if the manifest lacks a valid revision or SHA-256 for any file
    (authoritative hashes are never guessed), or lists any 14B / LoRA / alternative-VAE style file;
  * the exact byte size AND the SHA-256 of the whole file are checked against the manifest;
  * on any mismatch the file is DELETED and the run STOPS (exit code 4);
  * interrupted downloads resume from a `.part` file via HTTP Range requests, and a resumed file is
    trusted only after the SHA-256 of the complete file matches (a corrupt prefix cannot survive).

Exit codes: 0 ok | 1 --verify-only found a problem | 2 refused before downloading (manifest, disk space)
            3 download incomplete (network; partial kept for resume) | 4 verification failed (file deleted)

Tokens: an optional Hugging Face token is read ONLY from the HF_TOKEN environment variable (e.g. a Kaggle
Secret), sent only to the Hugging Face host (never to CDN redirect targets) and never printed.
Uses only the Python standard library (+ PyYAML). Weights must never be committed to Git.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import re
import shutil
import socket
import sys
import time
from http.client import HTTPException
from pathlib import Path
from urllib import error, parse, request

REPO_ROOT = Path(__file__).resolve().parents[1]
MANIFEST = REPO_ROOT / "workflows" / "comfyui" / "models.yaml"
DEFAULT_ENDPOINT = "https://huggingface.co"   # override with HF_ENDPOINT (e.g. a local test server)

CHUNK = 8 * 1024 * 1024
HASH_CHUNK = 16 * 1024 * 1024
MAX_ATTEMPTS = 8
PROGRESS_SECONDS = 15.0
SOCKET_TIMEOUT = 60.0
FREE_SPACE_MARGIN = 1 << 30                       # keep 1 GiB free after the downloads
RETRY_BASE_DELAY = float(os.environ.get("VIDSGEN_DL_RETRY_DELAY", "5"))

EXPECTED_ROLES = ("diffusion_model", "vae", "text_encoder")
ALLOWED_DIRS = {"models/diffusion_models", "models/vae", "models/text_encoders"}
FORBIDDEN_PATH = re.compile(r"14B|high_noise|low_noise|fun_|lora|wan_2\.1_vae|animate|s2v|inpaint|camera", re.I)
SHA256_RE = re.compile(r"^[0-9a-f]{64}$")
REVISION_RE = re.compile(r"^[0-9a-f]{40}$")


class DownloaderError(Exception):
    exit_code = 2


class Refused(DownloaderError):
    exit_code = 2


class IncompleteDownload(DownloaderError):
    exit_code = 3


class VerificationFailed(DownloaderError):
    exit_code = 4


# ------------------------------------------------------------------------------------ manifest
def load_manifest(path: Path) -> dict:
    try:
        import yaml
    except ImportError:
        sys.exit("PyYAML is required: pip install pyyaml")
    data = yaml.safe_load(Path(path).read_text())
    files = data.get("files") or []
    for f in files:
        for key in ("role", "repo", "path", "comfyui_dir", "size_bytes"):
            if key not in f:
                sys.exit(f"Manifest entry is missing '{key}': {f}")
    return data


def validate_for_download(manifest: dict) -> list[str]:
    """Everything that must hold BEFORE any byte is downloaded. Returns a list of problems."""
    problems: list[str] = []
    files = manifest.get("files") or []
    roles = [f.get("role") for f in files]
    if sorted(roles) != sorted(EXPECTED_ROLES):
        problems.append(f"manifest must list exactly the roles {list(EXPECTED_ROLES)} once each, found {roles}")
    for f in files:
        name = f"{f.get('role')} ({Path(str(f.get('path'))).name})"
        if not REVISION_RE.match(str(f.get("revision", ""))):
            problems.append(f"{name}: missing or invalid pinned `revision` (needs a 40-character commit id)")
        if not SHA256_RE.match(str(f.get("sha256", ""))):
            problems.append(f"{name}: authoritative `sha256` is missing or invalid (64 lowercase hex characters); "
                            "refusing to download a file whose hash is unknown")
        if not str(f.get("path", "")).startswith("split_files/") or ".." in str(f.get("path", "")):
            problems.append(f"{name}: path must be under split_files/")
        if FORBIDDEN_PATH.search(str(f.get("path", ""))):
            problems.append(f"{name}: path matches a file type this project must not download (14B/LoRA/alternative VAE)")
        if f.get("comfyui_dir") not in ALLOWED_DIRS:
            problems.append(f"{name}: comfyui_dir must be one of {sorted(ALLOWED_DIRS)}")
        if not isinstance(f.get("size_bytes"), int) or f["size_bytes"] <= 0:
            problems.append(f"{name}: size_bytes must be a positive integer")
    return problems


def gb(n: int) -> str:
    return f"{n / 1e9:.1f} GB"


def gib(n: float) -> str:
    return f"{n / 2**30:.2f} GiB"


def plan(manifest: dict, comfyui_dir: Path) -> list[dict]:
    rows = []
    for f in manifest["files"]:
        dest = comfyui_dir / f["comfyui_dir"] / Path(f["path"]).name
        size = dest.stat().st_size if dest.is_file() else None
        status = "present" if size == f["size_bytes"] else ("wrong size" if size is not None else "missing")
        rows.append({**f, "dest": dest, "status": status})
    return rows


# ------------------------------------------------------------------------------------ hashing
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while chunk := fh.read(HASH_CHUNK):
            h.update(chunk)
    return h.hexdigest()


def verify_file(path: Path, row: dict, label: str = "") -> tuple[bool, str]:
    """(ok, reason). Checks exact size first (cheap), then the SHA-256 of the whole file."""
    if not path.is_file():
        return False, "file does not exist"
    size = path.stat().st_size
    if size != row["size_bytes"]:
        return False, f"size {size:,} bytes != expected {row['size_bytes']:,}"
    t0 = time.monotonic()
    print(f"  {label}hashing {gib(size)} (SHA-256)...", flush=True)
    digest = sha256_file(path)
    took = time.monotonic() - t0
    if digest != row["sha256"]:
        return False, f"SHA-256 {digest} != expected {row['sha256']}"
    print(f"  {label}SHA-256 OK ({took:.0f}s)", flush=True)
    return True, "ok"


# ------------------------------------------------------------------------------------ network
class _NoRedirect(request.HTTPRedirectHandler):
    def redirect_request(self, *args, **kwargs):  # we follow redirects ourselves, to control headers
        return None


def _open_range(url: str, start: int, token: str, trusted_netloc: str):
    """GET url (optionally from byte `start`). Redirects are followed manually so the Authorization
    header is only ever sent to the Hugging Face host, never to a CDN host."""
    opener = request.build_opener(_NoRedirect)
    for _hop in range(6):
        headers = {"User-Agent": "vidsgen-model-downloader/2"}
        if start:
            headers["Range"] = f"bytes={start}-"
        if token and parse.urlsplit(url).netloc == trusted_netloc:
            headers["Authorization"] = f"Bearer {token}"
        try:
            return opener.open(request.Request(url, headers=headers), timeout=SOCKET_TIMEOUT)
        except error.HTTPError as e:
            location = e.headers.get("Location") if e.headers else None
            if e.code in (301, 302, 303, 307, 308) and location:
                url = parse.urljoin(url, location)
                e.close()
                continue
            raise
    raise IncompleteDownload("too many redirects")


def _retriable(e: BaseException) -> bool:
    if isinstance(e, error.HTTPError):
        return e.code in (408, 429) or e.code >= 500
    return isinstance(e, (error.URLError, socket.timeout, TimeoutError, ConnectionError, HTTPException, OSError))


class _Progress:
    def __init__(self, label: str, total: int, start: int):
        self.label, self.total, self.start = label, total, start
        self.t0 = self.last = time.monotonic()

    def tick(self, done: int, force: bool = False) -> None:
        now = time.monotonic()
        if not force and now - self.last < PROGRESS_SECONDS:
            return
        self.last = now
        rate = (done - self.start) / max(now - self.t0, 1e-6)
        eta = (self.total - done) / rate if rate > 0 else 0
        print(f"  {self.label}{gib(done)} / {gib(self.total)} ({100 * done / self.total:5.1f}%)  "
              f"{rate / 1e6:6.1f} MB/s  ETA {int(eta // 60)}m{int(eta % 60):02d}s", flush=True)


def _discard(*paths: Path) -> None:
    for p in paths:
        p.unlink(missing_ok=True)


def download_file(row: dict, endpoint: str, token: str, label: str = "") -> None:
    """Download one file into `<dest>.part`, resuming if possible, verify, then move it into place.
    Raises IncompleteDownload (network) or VerificationFailed (file deleted)."""
    dest: Path = row["dest"]
    part, meta = dest.with_name(dest.name + ".part"), dest.with_name(dest.name + ".part.json")
    expected = row["size_bytes"]
    want_meta = {"repo": row["repo"], "revision": row["revision"], "path": row["path"],
                 "size_bytes": expected, "sha256": row["sha256"]}
    url = f"{endpoint}/{row['repo']}/resolve/{row['revision']}/{parse.quote(row['path'])}"
    trusted = parse.urlsplit(endpoint).netloc

    # A partial file is resumed only if it provably belongs to this exact revision/hash and is not too big.
    if part.exists():
        try:
            same = json.loads(meta.read_text()) == want_meta
        except (OSError, ValueError):
            same = False
        if not same or part.stat().st_size > expected:
            print(f"  {label}discarding an unverifiable partial download", flush=True)
            _discard(part, meta)
    dest.parent.mkdir(parents=True, exist_ok=True)
    meta.write_text(json.dumps(want_meta))

    t0 = time.monotonic()
    attempt, last_have = 0, -1
    while True:
        have = part.stat().st_size if part.exists() else 0
        if have == expected:
            break
        if have > last_have:        # the previous try made progress: only consecutive failures count
            attempt = 0
        last_have = have
        attempt += 1
        if attempt > MAX_ATTEMPTS:
            raise IncompleteDownload(f"gave up after {MAX_ATTEMPTS} attempts without progress at "
                                     f"{have:,}/{expected:,} bytes (the partial file is kept; run again to resume)")
        try:
            resp = _open_range(url, have, token, trusted)
            status = resp.status
            length = resp.headers.get("Content-Length")
            if have and status == 206:
                m = re.match(r"bytes (\d+)-(\d+)/(\d+|\*)$", resp.headers.get("Content-Range", ""))
                if not m or int(m.group(1)) != have or (m.group(3) != "*" and int(m.group(3)) != expected):
                    raise VerificationFailed(f"server answered with an unexpected Content-Range "
                                             f"{resp.headers.get('Content-Range')!r}")
            if status == 200 and have:
                print(f"  {label}server ignored the Range request; restarting from byte 0", flush=True)
                have = 0
            if length is not None and int(length) != expected - have:
                raise VerificationFailed(f"server reports {int(length):,} bytes to send, expected {expected - have:,}: "
                                         f"this is not the file in the manifest")
            progress = _Progress(label, expected, have)
            if have:
                print(f"  {label}resuming at {gib(have)}", flush=True)
            with open(part, "ab" if have else "wb") as fh:
                done = have
                while chunk := resp.read(CHUNK):
                    done += len(chunk)
                    if done > expected:
                        raise VerificationFailed("server sent more bytes than the manifest size")
                    fh.write(chunk)
                    progress.tick(done)
                fh.flush()
                os.fsync(fh.fileno())
            resp.close()
            progress.tick(done, force=True)
            if done < expected:   # http.client returns silently when a server closes early
                print(f"  {label}connection closed early at {gib(done)}; retrying ({attempt}/{MAX_ATTEMPTS})", flush=True)
                time.sleep(min(RETRY_BASE_DELAY * 2 ** (attempt - 1), 60))
        except VerificationFailed:
            _discard(part, meta)
            raise
        except error.HTTPError as e:
            if e.code == 416:   # asked for bytes beyond the end: our partial file cannot be trusted
                _discard(part, meta)
                meta.write_text(json.dumps(want_meta))
                continue
            if _retriable(e):
                print(f"  {label}HTTP {e.code}; retrying ({attempt}/{MAX_ATTEMPTS})", flush=True)
                time.sleep(min(RETRY_BASE_DELAY * 2 ** (attempt - 1), 60))
                continue
            hint = " (check HF_TOKEN / access)" if e.code in (401, 403) else ""
            raise Refused(f"HTTP {e.code} for {row['repo']}@{row['revision'][:12]}/{row['path']}{hint}") from None
        except (OSError, HTTPException) as e:
            if isinstance(e, IncompleteDownload):
                raise
            print(f"  {label}connection problem ({type(e).__name__}); retrying ({attempt}/{MAX_ATTEMPTS})", flush=True)
            time.sleep(min(RETRY_BASE_DELAY * 2 ** (attempt - 1), 60))

    dl_time = time.monotonic() - t0
    size = part.stat().st_size
    if size != expected:
        _discard(part, meta)
        raise VerificationFailed(f"size {size:,} bytes != expected {expected:,}; file deleted")
    print(f"  {label}download complete ({dl_time:.0f}s); size {size:,} bytes OK", flush=True)
    ok, why = verify_file(part, row, label)
    if not ok:
        _discard(part, meta)
        raise VerificationFailed(f"{why}; the downloaded file was deleted")
    os.replace(part, dest)
    meta.unlink(missing_ok=True)


# ------------------------------------------------------------------------------------ commands
def _print_plan(manifest: dict, rows: list[dict], comfyui_dir: Path, manifest_path: Path) -> list[dict]:
    todo = [r for r in rows if r["status"] != "present"]
    print(f"Model: {manifest.get('model')}   manifest: {manifest_path}")
    print(f"ComfyUI directory: {comfyui_dir}\n")
    for r in rows:
        print(f"  [{r['status']:>10}] {r['role']:<16} {gb(r['size_bytes']):>8}  {r['repo']}/{r['path']}")
        if r.get("revision") or r.get("sha256"):
            print(f"               revision {str(r.get('revision', '?'))[:12]}  sha256 {str(r.get('sha256', '?'))[:16]}...")
        print(f"               -> {r['dest']}")
    print(f"\nTo download: {len(todo)} file(s), {gb(sum(r['size_bytes'] for r in todo))}")
    return todo


def cmd_verify_only(rows: list[dict]) -> int:
    bad = 0
    for i, r in enumerate(rows, 1):
        print(f"[{i}/{len(rows)}] {r['role']}: {r['dest']}")
        ok, why = verify_file(r["dest"], r, label="")
        print(f"  [{'PASS' if ok else 'FAIL'}] {why if not ok else 'size and SHA-256 match the manifest'}")
        bad += not ok
    print("\n" + ("ALL MODEL FILES VERIFIED (size + SHA-256)" if not bad else f"{bad} file(s) FAILED verification"))
    return 0 if not bad else 1


def cmd_execute(manifest: dict, rows: list[dict], comfyui_dir: Path) -> int:
    endpoint = os.environ.get("HF_ENDPOINT", DEFAULT_ENDPOINT).rstrip("/")
    token = os.environ.get("HF_TOKEN", "").strip()
    print(f"Source: {endpoint}   token: {'provided via HF_TOKEN (not shown)' if token else 'none (public repositories)'}")

    # Pre-flight: nothing is downloaded unless the whole set fits on disk.
    need = 0
    for r in rows:
        part = r["dest"].with_name(r["dest"].name + ".part")
        have = part.stat().st_size if part.exists() else 0
        if r["status"] != "present":
            need += max(r["size_bytes"] - have, 0)
    anchor = comfyui_dir
    while not anchor.exists():
        anchor = anchor.parent
    free = shutil.disk_usage(anchor).free
    print(f"Disk: {gib(free)} free at {anchor}; up to {gib(need)} still to download (+{gib(FREE_SPACE_MARGIN)} margin)")
    if free < need + FREE_SPACE_MARGIN:
        raise Refused(f"not enough free disk space ({gib(free)}) for {gib(need)} of downloads; nothing downloaded")

    for i, r in enumerate(rows, 1):
        label = f"[{i}/{len(rows)} {r['role']}] "
        print(f"\n{label}{r['repo']}@{r['revision'][:12]}/{r['path']}\n  destination: {r['dest']}", flush=True)
        if r["dest"].is_file():
            ok, why = verify_file(r["dest"], r, label)
            if ok:
                print(f"  {label}already present and verified; skipping", flush=True)
                continue
            print(f"  {label}existing file is not the expected one ({why}); deleting it and downloading again", flush=True)
            r["dest"].unlink()
        download_file(r, endpoint, token, label)
        print(f"  {label}VERIFIED and installed: {r['dest']}", flush=True)

    print(f"\nALL {len(rows)} MODEL FILES DOWNLOADED AND VERIFIED (exact size + SHA-256, pinned revisions)")
    return 0


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("--comfyui-dir", type=Path, default=Path("ComfyUI"),
                    help="Directory whose models/ subfolders receive the files. Default: ./ComfyUI")
    ap.add_argument("--manifest", type=Path, default=MANIFEST)
    mode = ap.add_mutually_exclusive_group()
    mode.add_argument("--execute", action="store_true", help="Actually download and verify. Without it: dry run only.")
    mode.add_argument("--verify-only", action="store_true", help="Hash the files already on disk; no network, no changes.")
    args = ap.parse_args(argv)

    manifest = load_manifest(args.manifest)
    comfyui_dir = args.comfyui_dir.resolve()
    rows = plan(manifest, comfyui_dir)
    try:
        if args.verify_only:
            return cmd_verify_only(rows)
        todo = _print_plan(manifest, rows, comfyui_dir, args.manifest)
        problems = validate_for_download(manifest)
        if not args.execute:
            if problems:
                print("\nNOTE: --execute would refuse to run until the manifest is fixed:")
                for p in problems:
                    print(f"  - {p}")
            print("\nDRY RUN: nothing was downloaded. Re-run with --execute to download.")
            return 0
        if problems:
            raise Refused("manifest not acceptable for download:\n  - " + "\n  - ".join(problems))
        if not todo:
            print("All model files are present; verifying them.")
        return cmd_execute(manifest, rows, comfyui_dir)
    except DownloaderError as e:
        print(f"\nSTOPPED: {e}", file=sys.stderr)
        return e.exit_code


if __name__ == "__main__":
    raise SystemExit(main())
''')
print('wrote', Path(TOOLS, 'download_models.py'), Path(TOOLS, 'download_models.py').stat().st_size, 'bytes')

In [ ]:
# Embedded verbatim from scripts/wan22_gpu_test.py, so this notebook is self-contained. A test keeps this copy identical.
Path(TOOLS, 'wan22_gpu_test.py').write_text(r'''#!/usr/bin/env python3
"""GPU-host tooling for the Wan 2.2 TI2V-5B test on Kaggle (Phase 5B). Used by
notebooks/kaggle_comfyui_wan22.ipynb; every piece is also unit-tested without a GPU.

Subcommands
  preflight       The gates that must pass before anything is installed or downloaded: Python version, a
                  non-persistent install location with enough free disk, GPU model / VRAM / CUDA-PyTorch
                  compatibility, system RAM, internet, and the authoritative Hugging Face metadata
                  (pinned revision, size, SHA-256) cross-checked against the manifest. No downloads.
  gpu-probe       GPU model, VRAM, CUDA/driver/PyTorch compatibility. Runs in its own process so the
                  notebook kernel never holds a CUDA context (that would steal VRAM from ComfyUI).
  install-comfyui Fetch ONE pinned ComfyUI commit, create a venv (--system-site-packages --without-pip: it
                  reuses the host's PyTorch and the host's pip), install ComfyUI's requirements under a
                  constraints file that pins torch/torchvision/torchaudio/numpy to what is already installed,
                  then verify they are the host's identical copies and that the host's own Python
                  environment is unchanged.
  serve / stop    Start ComfyUI in the background, bound to 127.0.0.1 ONLY, on one GPU, with a allow-listed
                  environment (no tokens or other credentials); verify it listens on loopback only. Stop it again.
  check-templates Validate BOTH Phase 4 workflow templates (text-to-video and image-to-video) against the
                  live server's own /object_info: every node class present, every input name, type, range and
                  combo value (including the model filenames) accepted.
  smoke-t2v       Run a small text-to-video generation through the existing VIDSGEN ComfyUIProvider and
                  the existing Phase 4 workflow, measure VRAM/RAM/time, and verify the MP4. Exit code 0
                  only if a real video was generated AND decodes with the requested frames and size.
  verify-mp4      Decode an MP4 and check frames, size, codec and that it is not blank.

Nothing here opens a public tunnel, downloads model weights, or prints secrets.
"""

from __future__ import annotations

import argparse
import asyncio
import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

GIB = 1024**3
# Used for pip/git style helper processes: drop anything that looks like a credential (best effort).
SECRET_ENV = re.compile(r"TOKEN|SECRET|PASSWORD|PASSWD|PASS$|ASKPASS|API_?KEY|ACCESS_?KEY|PRIVATE|CREDENTIAL|_KEY(_\d+)?$|"
                        r"KAGGLE_KEY|KAGGLE_USERNAME|^GIT_CONFIG_(KEY|VALUE)", re.I)
# The ComfyUI SERVER gets an ALLOWLIST instead: a denylist cannot be trusted to catch every credential name.
SERVER_ENV_ALLOW = re.compile(r"^(PATH|HOME|USER|LOGNAME|LANG|LANGUAGE|LC_[A-Z_]+|TERM|TZ|SHELL|LD_LIBRARY_PATH|LIBRARY_PATH|"
                              r"NVIDIA_[A-Z_]+|CUDA_[A-Z_]+|OMP_[A-Z_]+|MKL_[A-Z_]+|PYTORCH_[A-Z_]+|TORCH[A-Z_]*|NCCL_[A-Z_]+|"
                              r"CONDA_[A-Z_]+|VIRTUAL_ENV|PYTHON[A-Z_]*|TMPDIR|XDG_[A-Z_]+|HF_HOME|PIP_CACHE_DIR)$")
REDACT = [(re.compile(r"Bearer\s+\S+", re.I), "Bearer [redacted]"), (re.compile(r"hf_[A-Za-z0-9]{20,}"), "[redacted]"),
          (re.compile(r"(?i)(token|secret|password)=\S+"), r"\1=[redacted]")]
TORCH_PACKAGES = ("torch", "torchvision", "torchaudio")
PINNED_PACKAGES = (*TORCH_PACKAGES, "numpy")
# Needed by the existing VIDSGEN ComfyUIProvider (httpx, websockets) and the Phase 4 schema validator (starlette, uvicorn).
EXTRA_REQUIREMENTS = ["httpx>=0.27,<1.0", "websockets>=13", "uvicorn>=0.30", "starlette"]


def redact(text: str) -> str:
    for rx, repl in REDACT:
        text = rx.sub(repl, text)
    return text


def clean_env(base: dict[str, str] | None = None, **extra: str) -> dict[str, str]:
    """A copy of the environment without anything that looks like a credential."""
    env = {k: v for k, v in (base if base is not None else os.environ).items() if not SECRET_ENV.search(k)}
    env.update(extra)
    return env


def server_env(base_dir: Path, gpu: str | None, environ: dict[str, str] | None = None) -> dict[str, str]:
    """Environment for the ComfyUI server: only allow-listed variables, caches inside the base dir, one GPU."""
    src = environ if environ is not None else os.environ
    env = {k: v for k, v in src.items() if SERVER_ENV_ALLOW.match(k) and not SECRET_ENV.search(k)}
    env.update(cache_env(base_dir))
    if gpu not in (None, "", "none"):
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return env


def cache_env(base_dir: Path) -> dict[str, str]:
    """Send every cache/temp location to the (non-persistent) base directory."""
    cache = base_dir / "cache"
    for sub in ("pip", "hf", "torch", "xdg", "tmp"):
        (cache / sub).mkdir(parents=True, exist_ok=True)
    return {"PIP_CACHE_DIR": str(cache / "pip"), "HF_HOME": str(cache / "hf"), "TORCH_HOME": str(cache / "torch"),
            "XDG_CACHE_HOME": str(cache / "xdg"), "TMPDIR": str(cache / "tmp"), "PYTHONDONTWRITEBYTECODE": "1"}


def emit(obj) -> None:
    print(json.dumps(obj, indent=1, default=str))


# ===================================================================================== gpu-probe
def gpu_probe(gpu: int) -> dict:
    out: dict = {"nvidia_smi": None, "devices": [], "torch": None}
    try:
        p = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,driver_version,compute_cap",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30)
        if p.returncode == 0:
            out["nvidia_smi"] = [dict(zip(("index", "name", "memory_total_mib", "memory_used_mib", "driver", "compute_cap"),
                                          [x.strip() for x in line.split(",")])) for line in p.stdout.strip().splitlines()]
        banner = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout
        m = re.search(r"CUDA Version:\s*([\d.]+)", banner)
        out["driver_cuda"] = m.group(1) if m else None
    except (OSError, subprocess.SubprocessError) as e:
        out["nvidia_smi_error"] = f"{type(e).__name__}: {e}"
    try:
        import torch
    except ImportError as e:
        out["torch_error"] = str(e)
        return out
    out["torch"] = {"version": torch.__version__, "cuda_build": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
                    "arch_list": torch.cuda.get_arch_list() if torch.cuda.is_available() else []}
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            pr = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            out["devices"].append({"index": i, "name": pr.name, "capability": f"{pr.major}.{pr.minor}",
                                   "total_bytes": total, "free_bytes": free})
        try:
            a = torch.randn(256, 256, device=f"cuda:{gpu}", dtype=torch.float16)
            out["fp16_matmul_ok"] = bool(torch.isfinite(a @ a).all().item())
        except Exception as e:  # noqa: BLE001
            out["fp16_matmul_ok"] = False
            out["fp16_error"] = f"{type(e).__name__}: {str(e)[:200]}"
        try:
            out["bf16_native"] = bool(torch.cuda.is_bf16_supported(including_emulation=False))
        except TypeError:
            out["bf16_native"] = bool(torch.cuda.is_bf16_supported())
    return out


# ===================================================================================== preflight
DEFAULT_ENDPOINT = "https://huggingface.co"


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *a, **k):
        return None


def _hf_request(url: str, method: str = "GET", form: dict | None = None, token: str = "", timeout: float = 30.0):
    """Returns (status, headers, body). Redirects are NOT followed (a HEAD on /resolve/ answers 302 with the
    metadata headers we need, so no file content is ever transferred)."""
    headers = {"User-Agent": "vidsgen-preflight/1"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    data = urllib.parse.urlencode(form).encode() if form else None
    req = urllib.request.Request(url, data=data, method=method, headers=headers)
    try:
        with urllib.request.build_opener(_NoRedirect).open(req, timeout=timeout) as r:
            return r.status, r.headers, r.read()
    except urllib.error.HTTPError as e:
        return e.code, e.headers, e.read() if method != "HEAD" else b""


def hf_metadata_check(endpoint: str, row: dict, token: str = "") -> tuple[str, str]:
    """Compare the live, authoritative metadata for one model file with the manifest.
    SHA-256 = the Git-LFS object id, read from two independent sources that must agree. The Xet hash and the
    Git blob id are different identifiers and are never used."""
    repo, rev, path = row["repo"], row["revision"], row["path"]
    name = Path(path).name
    try:
        st, _, body = _hf_request(f"{endpoint}/api/models/{repo}/revision/{rev}", token=token)
        if st != 200 or json.loads(body).get("sha") != rev:
            return "FAIL", f"{name}: pinned revision {rev[:12]} of {repo} was not found (HTTP {st})"
        st, _, body = _hf_request(f"{endpoint}/api/models/{repo}/paths-info/{rev}", "POST",
                                  {"paths": path, "expand": "true"}, token)
        items = json.loads(body) if st == 200 else []
        lfs = (items[0].get("lfs") or {}) if items else {}
        st2, hdrs, _ = _hf_request(f"{endpoint}/{repo}/resolve/{rev}/{urllib.parse.quote(path)}", "HEAD", token=token)
        etag = (hdrs.get("x-linked-etag") or "").strip('"')
        size_head = hdrs.get("x-linked-size")
    except (OSError, ValueError, urllib.error.URLError) as e:
        return "FAIL", f"{name}: could not read Hugging Face metadata ({type(e).__name__})"
    problems = []
    if lfs.get("oid") != row["sha256"]:
        problems.append(f"paths-info SHA-256 {lfs.get('oid')} != manifest {row['sha256']}")
    if etag != row["sha256"]:
        problems.append(f"x-linked-etag {etag or None} != manifest {row['sha256']}")
    if lfs.get("size") != row["size_bytes"] or (size_head and int(size_head) != row["size_bytes"]):
        problems.append(f"size {lfs.get('size')}/{size_head} != manifest {row['size_bytes']}")
    if problems:
        return "FAIL", f"{name}: authoritative metadata disagrees with the manifest: " + "; ".join(problems)
    return "PASS", f"{name}: revision {rev[:12]}, {row['size_bytes']:,} bytes, SHA-256 {row['sha256'][:16]}... confirmed by 2 sources"


def disk_requirement(manifest: dict, comfy_data: Path, overhead_gib: float) -> tuple[int, int]:
    """(bytes of model files still to download, total bytes needed incl. overhead for packages/outputs/temp)."""
    remaining = 0
    for f in manifest["files"]:
        dest = comfy_data / f["comfyui_dir"] / Path(f["path"]).name
        part = dest.with_name(dest.name + ".part")
        if dest.is_file() and dest.stat().st_size == f["size_bytes"]:
            continue
        remaining += f["size_bytes"] - (part.stat().st_size if part.is_file() else 0)
    return remaining, remaining + int(overhead_gib * GIB)


def gpu_checks(probe: dict, gpu: int, min_vram_gib: float, allow_no_gpu: bool) -> list[tuple[str, str, str]]:
    out: list[tuple[str, str, str]] = []
    devs = probe.get("devices") or []
    t = probe.get("torch") or {}
    if not devs:
        why = probe.get("torch_error") or "PyTorch sees no CUDA GPU (set the notebook Accelerator to GPU T4 x2)"
        return [("gpu_detected", "WARN" if allow_no_gpu else "FAIL", why)]
    names = "; ".join(f"GPU{d['index']}: {d['name']} (sm_{d['capability'].replace('.', '')}, {d['total_bytes'] / GIB:.1f} GiB)" for d in devs)
    out.append(("gpu_detected", "PASS", names))
    out.append(("gpu_selection", "INFO", f"{len(devs)} GPU(s) visible. VRAM is NOT pooled: ComfyUI will use GPU {gpu} only"
                                         + (" (the other GPU stays idle)" if len(devs) > 1 else "")))
    d = next((x for x in devs if x["index"] == gpu), None)
    if d is None:
        return out + [("gpu_selected", "FAIL", f"GPU {gpu} is not available (visible: {[x['index'] for x in devs]})")]
    vram = d["total_bytes"] / GIB
    out.append(("gpu_vram", "PASS" if vram >= min_vram_gib else "WARN" if vram >= min_vram_gib - 2 else "FAIL",
                f"GPU {gpu}: {vram:.1f} GiB total, {d['free_bytes'] / GIB:.1f} GiB free (need >= {min_vram_gib:g} GiB; the fp16 5B model alone is ~9.3 GiB)"))
    cap = d["capability"].replace(".", "")
    in_arch = any(a.endswith("_" + cap) for a in t.get("arch_list", []))
    out.append(("torch_supports_gpu", "PASS" if in_arch else "FAIL",
                f"sm_{cap} {'is' if in_arch else 'is NOT'} in this PyTorch build's architectures {t.get('arch_list')}"))
    out.append(("fp16_kernel", "PASS" if probe.get("fp16_matmul_ok") else "FAIL",
                "a small fp16 matmul ran on the GPU" if probe.get("fp16_matmul_ok") else probe.get("fp16_error", "fp16 matmul failed")))
    build, drv = t.get("cuda_build"), probe.get("driver_cuda")
    if build and drv:
        ok = tuple(map(int, build.split(".")[:2])) <= tuple(map(int, drv.split(".")[:2]))
        out.append(("cuda_driver", "PASS" if ok else "WARN", f"PyTorch {t.get('version')} built for CUDA {build}; driver supports CUDA {drv}"
                                                              + ("" if ok else " (older driver: relies on minor-version compatibility)")))
    else:
        out.append(("cuda_driver", "WARN", f"PyTorch CUDA build {build}, driver CUDA {drv}: cannot compare"))
    out.append(("bf16", "INFO", f"native bf16: {probe.get('bf16_native')} (without it ComfyUI uses fp16/fp32; black frames would indicate fp16 overflow)"))
    return out


def preflight(base: Path, manifest_path: Path, gpu: int, min_vram_gib: float = 14.0, min_ram_gib: float = 24.0,
              overhead_gib: float = 8.0, offline: bool = False, allow_no_gpu: bool = False,
              repos: tuple[str, ...] = (), probe: dict | None = None, endpoint: str | None = None,
              pypi_url: str = "https://pypi.org/simple/pip/") -> dict:
    import yaml

    checks: list[dict] = []

    def add(name: str, status: str, detail: str) -> None:
        checks.append({"name": name, "status": status, "detail": detail})

    manifest = yaml.safe_load(Path(manifest_path).read_text())
    base = Path(base)
    add("python", "PASS" if sys.version_info >= (3, 11) else "FAIL",
        f"Python {sys.version.split()[0]} (the VIDSGEN backend code needs >= 3.11)")
    resolved = str(base.resolve()) if base.exists() else str(base)
    if resolved.startswith("/kaggle/working"):
        add("install_location", "FAIL", f"{base} is inside /kaggle/working, which is saved as notebook output and size-capped. Use /kaggle/tmp.")
    else:
        anchor = base
        while not anchor.exists() and anchor != anchor.parent:
            anchor = anchor.parent
        writable = os.access(anchor, os.W_OK)
        add("install_location", "PASS" if writable else "FAIL", f"{base} (non-persistent; nearest existing folder {anchor} is {'writable' if writable else 'NOT writable'})")
        remaining, need = disk_requirement(manifest, base / "comfy-data", overhead_gib)
        free = shutil.disk_usage(anchor).free
        add("disk_space", "PASS" if free >= need else "FAIL",
            f"{free / GIB:.1f} GiB free at {anchor}; need {need / GIB:.1f} GiB = {remaining / GIB:.1f} GiB of models still to download + "
            f"{overhead_gib:g} GiB for packages, outputs, temp files and margin")
    for name, st, detail in gpu_checks(probe if probe is not None else gpu_probe(gpu), gpu, min_vram_gib, allow_no_gpu):
        add(name, st, detail)
    try:
        mem = dict((k, int(v.split()[0]) * 1024) for k, v in (l.split(":", 1) for l in Path("/proc/meminfo").read_text().splitlines()))
        total, avail = mem["MemTotal"] / GIB, mem["MemAvailable"] / GIB
        add("system_ram", "PASS" if total >= min_ram_gib else "WARN" if total >= 0.8 * min_ram_gib else "FAIL",
            f"{total:.1f} GiB total, {avail:.1f} GiB available (offloaded weights live in RAM; need >= {min_ram_gib:g} GiB)")
    except (OSError, KeyError, ValueError):
        add("system_ram", "WARN", "could not read /proc/meminfo")
    if offline:
        add("network", "INFO", "skipped (--offline)")
    else:
        token = os.environ.get("HF_TOKEN", "")
        ep = (endpoint or os.environ.get("HF_ENDPOINT", DEFAULT_ENDPOINT)).rstrip("/")
        for url in repos:
            p = subprocess.run(["git", "ls-remote", "--exit-code", url, "HEAD"], capture_output=True, text=True, timeout=90)
            add(f"github {url.rsplit('/', 1)[-1]}", "PASS" if p.returncode == 0 else "FAIL",
                "reachable" if p.returncode == 0 else redact(p.stderr.strip())[-160:])
        try:
            st, _, _ = _hf_request(pypi_url, timeout=20)
            add("pypi", "PASS" if st == 200 else "FAIL", f"HTTP {st}")
        except OSError as e:
            add("pypi", "FAIL", type(e).__name__)
        for row in manifest["files"]:
            add(f"hf_metadata {row['role']}", *hf_metadata_check(ep, row, token))
    counts = {k: sum(c["status"] == k for c in checks) for k in ("PASS", "WARN", "FAIL", "INFO")}
    return {"ok": counts["FAIL"] == 0, "counts": counts, "checks": checks}


# ===================================================================================== install
def git(args: list[str], cwd: Path, timeout: int = 600) -> str:
    p = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed: {redact(p.stderr.strip())[-400:]}")
    return p.stdout.strip()


def checkout_commit(repo_url: str, commit: str, dest: Path) -> str:
    """Make `dest` a checkout of exactly `commit` (fetching only that commit). Returns the verified HEAD."""
    if (dest / ".git").exists() and _safe_head(dest) == commit:
        return commit
    dest.mkdir(parents=True, exist_ok=True)
    if not (dest / ".git").exists():
        git(["init", "-q"], dest)
        git(["remote", "add", "origin", repo_url], dest)
    git(["fetch", "-q", "--depth", "1", "origin", commit], dest)
    git(["checkout", "-q", "--detach", "FETCH_HEAD"], dest)
    head = git(["rev-parse", "HEAD"], dest)
    if head != commit:
        raise RuntimeError(f"checked out {head}, expected {commit}")
    return head


def _safe_head(dest: Path) -> str | None:
    try:
        return git(["rev-parse", "HEAD"], dest)
    except (RuntimeError, OSError):
        return None


def comfyui_version(checkout: Path) -> str | None:
    m = re.search(r'__version__\s*=\s*"([^"]+)"', (checkout / "comfyui_version.py").read_text())
    return m.group(1) if m else None


def filter_requirements(text: str) -> tuple[list[str], list[str]]:
    """Split a requirements.txt into (kept, removed). PyTorch packages are removed because the host
    already provides a working, GPU-matched build that must not be replaced."""
    kept, removed = [], []
    for line in text.splitlines():
        name = re.split(r"[<>=!~;\[\s#]", line.strip(), maxsplit=1)[0].lower().replace("_", "-")
        if name in TORCH_PACKAGES:
            removed.append(line.strip())
        else:
            kept.append(line.rstrip())
    return kept, removed


def write_constraints(path: Path, versions: dict[str, str]) -> None:
    path.write_text("\n".join(f"{n}=={v}" for n, v in sorted(versions.items())) + "\n")


# --- The install virtual environment -------------------------------------------------------------------------------
# --system-site-packages  the venv can see the host's PyTorch/CUDA stack, which must NOT be reinstalled.
# --without-pip           Kaggle's Python 3.13 has no `ensurepip`, so a venv cannot bootstrap its own pip. The host's pip
#                         stays importable through system-site-packages, and `<venv>/bin/python -m pip` installs into the
#                         VENV (pip derives the target from sys.prefix), leaving the host's site-packages untouched.
VENV_FLAGS = ("--system-site-packages", "--without-pip")


def venv_python(venv: Path) -> Path:
    return venv / "bin" / "python"


def venv_includes_system_packages(venv: Path) -> bool:
    """True if pyvenv.cfg says the venv can see the host's site-packages."""
    try:
        for line in (venv / "pyvenv.cfg").read_text().splitlines():
            key, _, value = line.partition("=")
            if key.strip().lower() == "include-system-site-packages":
                return value.strip().lower() == "true"
    except OSError:
        pass
    return False


def create_venv(python: str, venv: Path, env: dict[str, str]) -> dict:
    """Create the install venv, or accept an existing one, so re-running the installer is safe.

    An existing venv is reused as it is (never modified or deleted), whether it has its own pip or not. It must be
    able to see the host's packages, otherwise it could never import the host's PyTorch."""
    if venv_python(venv).exists():
        if not venv_includes_system_packages(venv):
            raise RuntimeError(f"the existing virtual environment {venv} does not include the host's site-packages "
                               "(pyvenv.cfg: include-system-site-packages is not true), so it cannot see the host's PyTorch. "
                               "Nothing was changed: delete that folder and run the install again.")
        return {"created": False}
    cmd = [python, "-m", "venv", *VENV_FLAGS, str(venv)]
    p = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=300)
    if p.returncode != 0 or not venv_python(venv).exists():
        raise RuntimeError(f"could not create the virtual environment ({' '.join(cmd)}):\n" + redact(p.stdout + p.stderr)[-800:])
    return {"created": True, "command": cmd}


def pip_info(py: str, venv: Path, env: dict[str, str]) -> dict:
    """Which pip the venv will use. A venv made with --without-pip relies on the HOST's pip."""
    p = subprocess.run([py, "-m", "pip", "--version"], capture_output=True, text=True, env=env, timeout=120)
    m = re.match(r"pip (\S+) from (.+?) \(python", p.stdout) if p.returncode == 0 else None
    if not m:
        raise RuntimeError("pip is not available inside the virtual environment. It was created without its own pip "
                           "(Kaggle's Python 3.13 has no ensurepip), so it needs the HOST Python's pip, and none was found. "
                           "Install pip for the host Python, or create the venv with pip, then run the install again.\n"
                           + redact(p.stdout + p.stderr)[-400:])
    version, location = m.group(1), m.group(2)
    return {"version": version, "location": location, "source": "venv" if str(venv) in location else "host"}


# --- Proving that the host's environment is untouched ----------------------------------------------------------------
IMPORT_PROBE = (
    "import importlib,json,sys\n"
    "out={}\n"
    f"for n in {list(PINNED_PACKAGES)!r}:\n"
    " try:\n"
    "  m=importlib.import_module(n);out[n]=[str(getattr(m,'__version__',None)),getattr(m,'__file__',None)]\n"
    " except ImportError:\n"
    "  out[n]=None\n"
    " except Exception as e:\n"
    "  out[n]=['error',type(e).__name__]\n"
    "print(json.dumps({'python':sys.version.split()[0],'packages':out}))")

INVENTORY_PROBE = (
    "import importlib.metadata as m,json\n"
    "d={}\n"
    "for x in m.distributions():\n"
    " n=(x.metadata['Name'] or '').lower().replace('_','-')\n"
    " if n:d.setdefault(n,[]).append(x.version)\n"
    "print(json.dumps({k:sorted(v) for k,v in d.items()}))")


def import_probe(py: str, env: dict[str, str]) -> dict:
    """{'python': ..., 'packages': {name: [version, file] | None}} for the protected packages, as imported by `py`."""
    p = subprocess.run([py, "-c", IMPORT_PROBE], capture_output=True, text=True, env=env, timeout=300)
    if p.returncode != 0:
        raise RuntimeError(f"the protected packages cannot be probed by {py}:\n" + redact(p.stderr)[-600:])
    return json.loads(p.stdout.strip().splitlines()[-1])


def protected_packages_replaced(host: dict, seen: dict, venv: Path) -> list[str]:
    """Problems if the venv does not use EXACTLY the host's torch, torchvision, torchaudio and numpy. Compares like with
    like (the imported modules' versions and file locations), so 'same version, different copy' is caught too. A package
    the host does not have is not protected (there is nothing to replace)."""
    problems = []
    for name in PINNED_PACKAGES:
        h, s = host["packages"].get(name), seen["packages"].get(name)
        if h is None:
            continue
        if s is None:
            problems.append(f"{name} {h[0]} cannot be imported inside the venv")
            continue
        if s[0] != h[0]:
            problems.append(f"{name} changed: {h[0]} -> {s[0]}")
        if s[1] != h[1]:
            problems.append(f"the venv imports a different {name} ({s[1]}; host: {h[1]})")
        if s[1] and str(venv) in s[1]:
            problems.append(f"a second {name} was installed inside the venv")
    return problems


def host_inventory(python: str, env: dict[str, str], cwd: Path) -> dict[str, list[str]]:
    """Every distribution (name -> versions) the HOST Python can see, read by a separate process."""
    p = subprocess.run([python, "-c", INVENTORY_PROBE], capture_output=True, text=True, env=env, cwd=cwd, timeout=300)
    if p.returncode != 0:
        raise RuntimeError(f"cannot read the host's installed packages with {python}:\n" + redact(p.stderr)[-600:])
    return json.loads(p.stdout.strip().splitlines()[-1])


def inventory_changes(before: dict[str, list[str]], after: dict[str, list[str]]) -> list[str]:
    out = []
    for name in sorted(set(before) | set(after)):
        b, a = before.get(name), after.get(name)
        if b == a:
            continue
        if b is None:
            out.append(f"{name} was ADDED ({', '.join(a)})")
        elif a is None:
            out.append(f"{name} was REMOVED (was {', '.join(b)})")
        else:
            out.append(f"{name} changed: {', '.join(b)} -> {', '.join(a)}")
    return out


# --- Which version of each protected package must be pinned? ---------------------------------------------------------
# A Python environment can contain SEVERAL metadata records (dist-info) for one package, e.g. a stale numpy-1.26.4.dist-info next to
# the real numpy 2.1.3. Taking "the first of the sorted versions" picks the stale record, so the constraints file then told pip to
# install numpy==1.26.4 into the venv (a second NumPy, replacing the host's 2.1.3 for the venv). The pin must instead come from the
# package the host really IMPORTS, using the metadata record that describes that exact release.
PINS_PROBE = (
    "import importlib,importlib.metadata as m,json\n"
    "out={}\n"
    f"for n in {list(PINNED_PACKAGES)!r}:\n"
    " try:\n"
    "  mod=importlib.import_module(n);imp=[str(getattr(mod,'__version__',None)),getattr(mod,'__file__',None)]\n"
    " except ImportError:\n"
    "  imp=None\n"
    " except Exception as e:\n"
    "  imp=['error',type(e).__name__]\n"
    " ds=[[d.version,str(getattr(d,'_path',''))] for d in m.distributions() if (d.metadata['Name'] or '').lower().replace('_','-')==n]\n"
    " out[n]={'imported':imp,'metadata':ds}\n"
    "print(json.dumps(out))")


def _public(version: str) -> str:
    """'2.11.0+cu130' -> '2.11.0' (PyTorch's module version carries a local build tag that its metadata may omit)."""
    return str(version).split("+", 1)[0]


def select_pins(probe: dict) -> tuple[dict[str, str], dict[str, list[list[str]]]]:
    """(pins, conflicts) from the PINS_PROBE output.

    pins: name -> the version string pip knows for the package the host actually imports. It must match the imported module's
    release, otherwise the pin cannot be trusted and this raises (it never guesses). conflicts: packages with more than one
    metadata record, with their locations (reported so a stale record is visible, not silently ignored)."""
    pins: dict[str, str] = {}
    conflicts: dict[str, list[list[str]]] = {}
    for name in PINNED_PACKAGES:
        entry = probe.get(name) or {}
        imported, records = entry.get("imported"), entry.get("metadata") or []
        if len({v for v, _ in records}) > 1:
            conflicts[name] = records
        if imported is None:
            if records:                      # installed but not importable on the host: still keep pip from replacing it
                pins[name] = records[0][0]
            continue
        if imported[0] == "error":
            raise RuntimeError(f"{name} is installed on the host but importing it fails ({imported[1]}); refusing to continue")
        matching = [v for v, _ in records if _public(v) == _public(imported[0])]
        if not matching:
            raise RuntimeError(f"cannot establish which {name} the host uses: it imports {imported[0]} from {imported[1]}, "
                               f"but its installed-package records say {[v for v, _ in records] or 'nothing'}; refusing to guess a pin")
        pins[name] = matching[0]
    return pins, conflicts


def host_pins(python: str, env: dict[str, str], cwd: Path) -> tuple[dict[str, str], dict[str, list[list[str]]]]:
    """The versions to protect, taken from what the host's Python imports (see select_pins)."""
    p = subprocess.run([python, "-c", PINS_PROBE], capture_output=True, text=True, env=env, cwd=cwd, timeout=300)
    if p.returncode != 0:
        raise RuntimeError(f"cannot inspect the protected packages with {python}:\n" + redact(p.stderr)[-600:])
    return select_pins(json.loads(p.stdout.strip().splitlines()[-1]))


def install_comfyui(base: Path, repo_url: str, commit: str, expect_version: str | None, python: str = sys.executable) -> dict:
    base.mkdir(parents=True, exist_ok=True)
    cenv = cache_env(base)
    checkout = base / "ComfyUI"
    head = checkout_commit(repo_url, commit, checkout)
    version = comfyui_version(checkout)
    if expect_version and version != expect_version:
        raise RuntimeError(f"ComfyUI at {commit[:12]} reports version {version}, expected {expect_version}")
    dirty = git(["status", "--porcelain"], checkout)
    if dirty:
        raise RuntimeError("ComfyUI checkout has local modifications:\n" + dirty[:300])

    env = clean_env(None, **cenv)
    inventory_before = host_inventory(python, env, base)        # the host's packages, before we touch anything (change detection only)
    before, metadata_conflicts = host_pins(python, env, base)   # what to protect: the packages the host really imports
    if "torch" not in before:
        raise RuntimeError("PyTorch is not installed on this host; refusing to install one blindly")
    constraints = base / "constraints.txt"
    write_constraints(constraints, before)                      # torch/torchvision/torchaudio/numpy stay exactly as they are
    kept, removed = filter_requirements((checkout / "requirements.txt").read_text())
    (base / "requirements-comfyui.txt").write_text("\n".join(kept) + "\n")
    (base / "requirements-extra.txt").write_text("\n".join(EXTRA_REQUIREMENTS) + "\n")

    venv = base / "venv"
    venv_info = create_venv(python, venv, env)
    pip = pip_info(str(venv_python(venv)), venv, env)
    cmd = [str(venv_python(venv)), "-m", "pip", "install", "--no-cache-dir", "--disable-pip-version-check", "-q",
           "-c", str(constraints), "-r", str(base / "requirements-comfyui.txt"), "-r", str(base / "requirements-extra.txt")]
    t0 = time.monotonic()
    p = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=3600)
    if p.returncode != 0:
        raise RuntimeError("pip install failed (PyTorch and NumPy were protected by the constraints file):\n" + redact(p.stdout + p.stderr)[-1500:])

    host_seen, seen = import_probe(python, env), import_probe(str(venv_python(venv)), env)
    problems = protected_packages_replaced(host_seen, seen, venv)
    if problems:
        raise RuntimeError("the host's PyTorch/NumPy stack was replaced: " + "; ".join(problems)
                           + f" (pinned from the host: {', '.join(f'{n}=={v}' for n, v in sorted(before.items()))}"
                           + (f"; host packages with several metadata records: {sorted(metadata_conflicts)}" if metadata_conflicts else "") + ")")
    changes = inventory_changes(inventory_before, host_inventory(python, env, base))
    if changes:
        more = f" (+{len(changes) - 12} more)" if len(changes) > 12 else ""
        raise RuntimeError("the host's Python environment was modified by the install (it must stay untouched): "
                           + "; ".join(changes[:12]) + more)
    chk = subprocess.run([str(venv_python(venv)), "-m", "pip", "check"], capture_output=True, text=True, env=env, timeout=300)
    protected = {n: {"version": v[0], "file": v[1], "identical_to_host": True}
                 for n, v in seen["packages"].items() if host_seen["packages"].get(n)}
    return {"commit": head, "comfyui_version": version, "checkout": str(checkout), "venv": str(venv),
            "venv_created": venv_info["created"], "venv_flags": list(VENV_FLAGS), "pip": pip,
            "python": seen["python"], "torch": seen["packages"]["torch"][0], "torch_file": seen["packages"]["torch"][1],
            "torch_replaced": False, "protected_packages": protected,
            "host_environment_unchanged": True, "host_packages_checked": len(inventory_before),
            "pinned_by_constraints": before, "host_metadata_conflicts": metadata_conflicts, "requirements_removed": removed, "pip_seconds": round(time.monotonic() - t0),
            "pip_check": (chk.stdout + chk.stderr).strip()[-600:] or "No broken requirements found."}


# ===================================================================================== serve / stop
def listening_addresses(port: int, proc_root: Path = Path("/proc/net")) -> list[str]:
    """Local addresses that are in LISTEN state on `port` (parsed from /proc/net/tcp{,6})."""
    found = []
    for name, width in (("tcp", 8), ("tcp6", 32)):
        f = proc_root / name
        if not f.exists():
            continue
        for line in f.read_text().splitlines()[1:]:
            cols = line.split()
            if len(cols) < 4 or cols[3] != "0A":
                continue
            addr, hexport = cols[1].rsplit(":", 1)
            if int(hexport, 16) != port:
                continue
            if width == 8:
                found.append(".".join(str(int(addr[i:i + 2], 16)) for i in (6, 4, 2, 0)))
            else:
                found.append("::" if set(addr) == {"0"} else ("::1" if addr == "0" * 30 + "01" or addr == "00000000000000000000000001000000" else addr))
    return sorted(set(found))


def is_loopback(addr: str) -> bool:
    return addr.startswith("127.") or addr == "::1"


def http_get(url: str, timeout: float = 5.0) -> tuple[int, bytes]:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status, r.read()
    except urllib.error.HTTPError as e:
        return e.code, b""


def comfy_args(base: Path, port: int, gpu: str | None, extra: list[str]) -> list[str]:
    data = base / "comfy-data"
    args = ["--listen", "127.0.0.1", "--port", str(port), "--disable-auto-launch",
            "--base-directory", str(data), "--temp-directory", str(base / "comfy-temp")]
    if gpu not in (None, "", "none"):
        args += ["--cuda-device", str(gpu)]
    return args + list(extra)


def serve(base: Path, port: int, gpu: str | None, extra: list[str], wait: float) -> dict:
    checkout, venv = base / "ComfyUI", base / "venv"
    if not (checkout / "main.py").exists() or not venv_python(venv).exists():
        raise RuntimeError("ComfyUI is not installed in " + str(base) + " (run install-comfyui first)")
    for sub in ("models/diffusion_models", "models/text_encoders", "models/vae", "custom_nodes", "input", "output", "user"):
        (base / "comfy-data" / sub).mkdir(parents=True, exist_ok=True)
    (base / "comfy-temp").mkdir(parents=True, exist_ok=True)
    (base / "logs").mkdir(parents=True, exist_ok=True)
    pidfile, logfile = base / "logs" / "comfyui.pid", base / "logs" / "comfyui.log"
    stop(base, quiet=True)

    env = server_env(base, gpu)
    cmd = [str(venv_python(venv)), str(checkout / "main.py"), *comfy_args(base, port, gpu, extra)]
    with open(logfile, "wb") as log:   # the child keeps its own copy of the handle; the parent must not leak one
        proc = subprocess.Popen(cmd, cwd=checkout, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    pidfile.write_text(str(proc.pid))
    _CHILDREN[proc.pid] = proc
    deadline, ready = time.monotonic() + wait, False
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            stop(base, quiet=True)
            raise RuntimeError(f"ComfyUI exited with code {proc.returncode}:\n" + redact(tail(logfile, 40)))
        try:
            if http_get(f"http://127.0.0.1:{port}/system_stats")[0] == 200:
                ready = True
                break
        except OSError:
            pass
        time.sleep(1)
    if not ready:
        stop(base, quiet=True)
        raise RuntimeError(f"ComfyUI did not become ready within {wait:.0f}s:\n" + redact(tail(logfile, 40)))
    addrs = listening_addresses(port)
    if not addrs or not all(is_loopback(a) for a in addrs):
        stop(base, quiet=True)
        raise RuntimeError(f"ComfyUI is listening on {addrs}, not loopback only; stopped it")
    return {"pid": proc.pid, "port": port, "listening_on": addrs, "loopback_only": True, "log": str(logfile),
            "args": comfy_args(base, port, gpu, extra), "cuda_visible_devices": env.get("CUDA_VISIBLE_DEVICES")}


_CHILDREN: dict[int, subprocess.Popen] = {}   # servers started by this process, kept so they are reaped (no zombies)


def tail(path: Path, n: int) -> str:
    try:
        return "\n".join(path.read_text(errors="replace").splitlines()[-n:])
    except OSError:
        return ""


def stop(base: Path, quiet: bool = False) -> bool:
    """Stop the ComfyUI started by `serve` (SIGTERM to its process group, SIGKILL after ~15 s)."""
    pidfile = base / "logs" / "comfyui.pid"
    if not pidfile.exists():
        return False
    try:
        pid = int(pidfile.read_text())
    except ValueError:
        pidfile.unlink(missing_ok=True)
        return False
    proc = _CHILDREN.pop(pid, None)
    try:
        group = os.getpgid(pid)
        os.killpg(group, signal.SIGTERM)
        for _ in range(60):
            time.sleep(0.25)
            if (proc.poll() is not None) if proc is not None else not _alive(pid):
                break
        else:
            os.killpg(group, signal.SIGKILL)
    except OSError:
        pass                      # already gone
    if proc is not None:
        try:
            proc.wait(timeout=10)
        except subprocess.SubprocessError:
            pass
    pidfile.unlink(missing_ok=True)
    if not quiet:
        print("ComfyUI stopped")
    return True


def _alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


# ===================================================================================== MP4 verification
def probe_mp4(path: Path, frames: int | None = None, size: tuple[int, int] | None = None, fps: float | None = None) -> dict:
    """Open and fully decode an MP4 with PyAV. `problems` is empty only for a real, non-blank video."""
    import numpy as np

    path = Path(path)
    r: dict = {"path": str(path), "exists": path.is_file(), "bytes": path.stat().st_size if path.is_file() else 0,
               "opened": False, "problems": []}
    if not r["exists"] or r["bytes"] < 1024:
        r["problems"].append("file is missing or implausibly small")
        return r
    r["ftyp"] = path.read_bytes()[4:8] == b"ftyp"
    if not r["ftyp"]:
        r["problems"].append("not an MP4 (no ftyp box)")
    try:
        import av

        with av.open(str(path)) as c:
            r["opened"] = True
            r["container"] = c.format.name
            v = c.streams.video[0]
            cc = v.codec_context
            r.update(codec=cc.name, width=cc.width, height=cc.height, pix_fmt=cc.pix_fmt,
                     fps=float(v.average_rate) if v.average_rate else None,
                     duration_s=round(c.duration / 1_000_000, 3) if c.duration else None)
            means, stds, first, last = [], [], None, None
            for frame in c.decode(video=0):
                g = frame.to_ndarray(format="gray")
                means.append(float(g.mean()))
                stds.append(float(g.std()))
                small = g[::4, ::4].astype(np.int16)
                first = small if first is None else first
                last = small
            r["decoded_frames"] = len(means)
            if means:
                r["luma_mean_min_max"] = [round(min(means), 2), round(max(means), 2)]
                r["luma_std_max"] = round(max(stds), 2)
                r["first_last_mean_abs_diff"] = round(float(np.abs(first - last).mean()), 2)
    except Exception as e:  # noqa: BLE001
        r["problems"].append(f"cannot be decoded: {type(e).__name__}: {str(e)[:200]}")
        return r
    if "mp4" not in r["container"]:
        r["problems"].append(f"container is {r['container']}, not MP4")
    if r["codec"] != "h264":
        r["problems"].append(f"video codec is {r['codec']}, expected h264")
    if frames is not None and r["decoded_frames"] != frames:
        r["problems"].append(f"decoded {r['decoded_frames']} frames, expected {frames}")
    if size is not None and (r["width"], r["height"]) != tuple(size):
        r["problems"].append(f"size is {r['width']}x{r['height']}, expected {size[0]}x{size[1]}")
    if fps is not None and r["fps"] is not None and abs(r["fps"] - fps) > 0.01:
        r["problems"].append(f"fps is {r['fps']}, expected {fps}")
    if r["decoded_frames"] and r["luma_mean_min_max"][1] < 3.0:
        r["problems"].append("every frame is black (suspected fp16 numerical failure)")
    elif r["decoded_frames"] and r["luma_std_max"] < 1.0:
        r["problems"].append("every frame is a flat single colour (no image content)")
    return r


# ===================================================================================== metrics
class Sampler(threading.Thread):
    """Samples GPU memory (nvidia-smi), host RAM and the ComfyUI process' RSS while a job runs."""

    def __init__(self, gpu: int = 0, pid: int | None = None, interval: float = 0.5):
        super().__init__(daemon=True)
        self.gpu, self.pid, self.interval = gpu, pid, interval
        self._stop = threading.Event()
        self.gpu_available = shutil.which("nvidia-smi") is not None
        self.samples = 0
        self.vram_used_mib: list[int] = []
        self.gpu_util: list[int] = []
        self.vram_total_mib: int | None = None
        self.ram_available_min: int | None = None
        self.ram_total: int | None = None
        self.rss_peak_kib: int | None = None
        self.baseline_vram_mib: int | None = None

    def sample_once(self) -> None:
        self.samples += 1
        if self.gpu_available:
            try:
                p = subprocess.run(["nvidia-smi", "-i", str(self.gpu), "--query-gpu=memory.used,memory.total,utilization.gpu",
                                    "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5)
                used, total, util = [x.strip() for x in p.stdout.strip().split(",")[:3]]
                self.vram_used_mib.append(int(used))
                self.vram_total_mib = int(total)
                if util.isdigit():
                    self.gpu_util.append(int(util))
            except (OSError, ValueError, subprocess.SubprocessError):
                pass
        try:
            mem = dict((k, int(v.split()[0]) * 1024) for k, v in (l.split(":", 1) for l in Path("/proc/meminfo").read_text().splitlines()))
            self.ram_total = mem["MemTotal"]
            self.ram_available_min = min(self.ram_available_min or mem["MemAvailable"], mem["MemAvailable"])
        except (OSError, KeyError, ValueError):
            pass
        if self.pid:
            try:
                for line in Path(f"/proc/{self.pid}/status").read_text().splitlines():
                    if line.startswith("VmHWM:"):
                        self.rss_peak_kib = max(self.rss_peak_kib or 0, int(line.split()[1]))
            except (OSError, ValueError):
                pass

    def run(self) -> None:
        while not self._stop.is_set():
            self.sample_once()
            self._stop.wait(self.interval)

    def finish(self) -> dict:
        self._stop.set()
        self.join(timeout=10)
        self.sample_once()
        peak = max(self.vram_used_mib) if self.vram_used_mib else None
        return {
            "samples": self.samples,
            "gpu_sampling": "nvidia-smi" if self.vram_used_mib else "unavailable",
            "gpu_index": self.gpu,
            "vram_total_mib": self.vram_total_mib,
            "vram_baseline_mib": self.baseline_vram_mib,
            "vram_peak_used_mib": peak,
            "vram_peak_above_baseline_mib": (peak - self.baseline_vram_mib) if peak is not None and self.baseline_vram_mib is not None else None,
            "gpu_util_peak_pct": max(self.gpu_util) if self.gpu_util else None,
            "gpu_util_mean_pct": round(sum(self.gpu_util) / len(self.gpu_util)) if self.gpu_util else None,
            "ram_total_gib": round(self.ram_total / GIB, 2) if self.ram_total else None,
            "ram_peak_used_gib": round((self.ram_total - self.ram_available_min) / GIB, 2) if self.ram_total and self.ram_available_min is not None else None,
            "comfyui_rss_peak_gib": round(self.rss_peak_kib / 1024 / 1024, 2) if self.rss_peak_kib else None,
        }


# ===================================================================================== template check
def check_templates(url: str, backend_dir: Path, workflows_dir: Path) -> dict:
    """Reuse the Phase 4 code: WorkflowTemplates builds the real graphs, validate_graph checks them against
    the schemas that THIS server reports. Catches ComfyUI version drift and missing model files."""
    sys.path[:0] = [str(Path(backend_dir).resolve()), str(Path(backend_dir).resolve() / "tests")]
    from app.jobs.models import GenerationMode
    from app.workflows import GraphValues, WorkflowTemplates
    from comfyui_support import validate_graph

    wt = WorkflowTemplates(Path(workflows_dir))
    info: dict = {}
    missing: list[str] = []
    for cls in wt.node_classes:
        status, body = http_get(f"{url}/object_info/{cls}", timeout=30)
        data = json.loads(body) if status == 200 and body else {}
        if cls in data:
            info[cls] = data[cls]
        else:
            missing.append(cls)
    out: dict = {"server": url, "node_classes_required": wt.node_classes, "node_classes_missing": missing, "templates": {}}
    for mode in GenerationMode:
        values = GraphValues(prompt="check", negative_prompt="", width=832, height=480, num_frames=17, fps=24, steps=20,
                             guidance_scale=5.0, seed=1, filename_prefix="vidsgen/job_check",
                             image="vidsgen/job_check.png" if mode == GenerationMode.i2v else None)
        problems = validate_graph(wt.build(mode, values), info) if not missing else ["skipped: required node classes are missing"]
        out["templates"][mode.value] = {"problems": problems, "ok": not problems}
    out["ok"] = not missing and all(t["ok"] for t in out["templates"].values())
    return out


# ===================================================================================== smoke test
DEFAULT_PROMPT = ("A paper boat drifting down a rainy neon-lit city street at night, reflections rippling on wet asphalt, "
                  "slow cinematic camera movement")


async def run_provider(args, sampler: Sampler) -> dict:
    sys.path.insert(0, str(Path(args.backend_dir).resolve()))
    from app.jobs.models import GenerationMode
    from app.providers.base import GenerationRequest, ProviderError
    from app.providers.comfyui import ComfyUIProvider
    from app.workflows import WorkflowTemplates

    width, height = (int(x) for x in args.resolution.split("x"))
    prov = ComfyUIProvider(base_url=args.url, workflows=WorkflowTemplates(Path(args.workflows_dir)),
                           api_token=os.environ.get("COMFYUI_API_TOKEN", ""), timeout=60.0, poll_interval=2.0)
    res: dict = {"queued": False, "updates": [], "final_state": None, "error": None}
    try:
        h = await prov.health()
        res["health_before"] = {"ok": h.ok, "detail": h.detail, "checks": [{"name": c.name, "status": c.status, "detail": c.detail} for c in h.checks]}
        if not h.ok:
            res["error"] = {"code": "provider_not_ready", "message": h.detail}
            return res
        sampler.sample_once()
        sampler.baseline_vram_mib = sampler.vram_used_mib[-1] if sampler.vram_used_mib else None
        request = GenerationRequest(job_id="job_smoke_t2v", mode=GenerationMode.t2v, prompt=args.prompt, negative_prompt="",
                                    width=width, height=height, num_frames=args.frames, fps=args.fps, steps=args.steps,
                                    guidance_scale=args.cfg, seed=args.seed)
        t0 = time.monotonic()
        pid = await prov.submit(request)
        res["queued"], res["prompt_id"] = True, pid
        try:
            async with asyncio.timeout(args.timeout):
                last_beat = 0.0
                async for u in prov.progress(pid):
                    now = time.monotonic() - t0
                    res["updates"].append({"t": round(now, 2), "state": u.state, "percent": u.percent, "message": u.message})
                    if u.state == "failed":
                        res["error"] = {"code": u.error_code, "message": u.message}
                    if now - last_beat >= 10 or u.state in ("completed", "failed", "cancelled"):   # a heartbeat for long runs
                        last_beat = now
                        vram = f" | GPU memory {sampler.vram_used_mib[-1]} MiB" if sampler.vram_used_mib else ""
                        print(f"  [{now:6.0f}s] {u.state:<9} {u.percent:5.1f}%  {u.message}{vram}", file=sys.stderr, flush=True)
        except TimeoutError:
            await prov.cancel(pid)
            res["error"] = {"code": "timeout", "message": f"no result within {args.timeout}s; cancelled"}
        res["final_state"] = res["updates"][-1]["state"] if res["updates"] else None
        res["wall_seconds"] = round(time.monotonic() - t0, 1)
        if res["final_state"] == "completed":
            result = await prov.fetch_result(pid)
            Path(args.out).parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(result.path, args.out)
            result.cleanup()
            res["output"] = args.out
        await prov.release(pid)
        h2 = await prov.health()
        res["health_after"] = {"ok": h2.ok, "detail": h2.detail, "checks": [{"name": c.name, "status": c.status, "detail": c.detail} for c in h2.checks]}
    except ProviderError as e:
        res["error"] = {"code": e.code, "message": e.message}
    finally:
        await prov.aclose()
    return res


def step_timing(updates: list[dict]) -> dict:
    steps = [(u["t"], int(m.group(1)), int(m.group(2))) for u in updates if (m := re.match(r"step (\d+)/(\d+)", u["message"]))]
    out: dict = {"first_update_s": updates[0]["t"] if updates else None}
    if steps:
        out.update(sampler_steps_seen=len(steps), first_step_at_s=steps[0][0], last_step_at_s=steps[-1][0], steps_total=steps[-1][2])
        if steps[-1][1] > steps[0][1]:
            out["seconds_per_step"] = round((steps[-1][0] - steps[0][0]) / (steps[-1][1] - steps[0][1]), 2)
    return out


def evaluate(res: dict, probe: dict | None, sampler_stats: dict, require_gpu: bool) -> tuple[str, list[str]]:
    """PASS only if a real video came out. Queued/started/completed-without-a-file never counts."""
    reasons: list[str] = []
    if not res.get("queued"):
        reasons.append("the workflow was never queued: " + json.dumps(res.get("error")))
    elif res.get("final_state") != "completed":
        reasons.append(f"generation ended as {res.get('final_state')!r}: {json.dumps(res.get('error'))}")
    if res.get("final_state") == "completed":
        if probe is None:
            reasons.append("no output file was retrieved")
        else:
            reasons += probe["problems"]
    if require_gpu and sampler_stats["gpu_sampling"] == "unavailable":
        reasons.append("GPU memory could not be measured (nvidia-smi unavailable)")
    return ("PASS" if not reasons else "FAIL"), reasons


def smoke(args) -> int:
    """One smoke test; if it fails with an out-of-memory error and fallback arguments were given, restart
    ComfyUI once with them (e.g. --disable-dynamic-vram --disable-smart-memory) and try again."""
    code = smoke_once(args)
    fallback = getattr(args, "fallback_args", None) or []
    fallback = shlex.split(fallback) if isinstance(fallback, str) else list(fallback)
    if code == 0 or not fallback or not args.base:
        return code
    first = json.loads(Path(args.report).read_text())
    if (first.get("error") or {}).get("code") != "gpu_out_of_memory":
        return code
    print(f"\nOut of GPU memory with default settings. Restarting ComfyUI ONCE with {fallback} and retrying...", file=sys.stderr, flush=True)
    first_path = Path(args.report).with_name(Path(args.report).stem + "_attempt1.json")
    shutil.copyfile(args.report, first_path)
    stop(Path(args.base), quiet=True)
    info = serve(Path(args.base), int(getattr(args, "port", 8188)), str(args.gpu), fallback, wait=300)
    code = smoke_once(args)
    second = json.loads(Path(args.report).read_text())
    second["retry"] = {"first_attempt_report": str(first_path), "first_attempt_error": first.get("error"), "fallback_args": fallback,
                       "server_args": info["args"]}
    Path(args.report).write_text(json.dumps(second, indent=1, default=str))
    return code


def smoke_once(args) -> int:
    pid = None
    pidfile = Path(args.base) / "logs" / "comfyui.pid" if args.base else None
    if pidfile and pidfile.exists():
        pid = int(pidfile.read_text())
    sampler = Sampler(gpu=args.gpu, pid=pid)
    sampler.start()
    t_start = time.monotonic()
    try:
        res = asyncio.run(run_provider(args, sampler))
    except ImportError as e:   # e.g. run with the system python instead of the ComfyUI venv's python
        res = {"queued": False, "updates": [], "final_state": None, "error": {
            "code": "missing_dependency", "message": f"{e}. Run this command with the ComfyUI venv's python (<base>/venv/bin/python)."}}
    stats = sampler.finish()
    size = tuple(int(x) for x in args.resolution.split("x"))
    probe = probe_mp4(Path(args.out), frames=args.frames, size=size, fps=float(args.fps)) if res.get("output") else None
    verdict, reasons = evaluate(res, probe, stats, args.require_gpu)
    server_log = redact(tail(Path(args.base) / "logs" / "comfyui.log", 60)) if args.base and verdict == "FAIL" else None
    report = {
        "verdict": verdict, "reasons": reasons,
        "claims_checked": {"server_responded": res.get("health_before") is not None, "workflow_queued": res.get("queued", False),
                           "generation_completed": res.get("final_state") == "completed",
                           "mp4_exists": bool(probe and probe["exists"]), "mp4_decodes": bool(probe and probe["opened"]),
                           "frames_and_size_match": bool(probe and not any("frames" in p or "size" in p for p in probe["problems"])),
                           "not_blank": bool(probe and probe["opened"] and not any("black" in p or "flat" in p for p in probe["problems"]))},
        "request": {"mode": "t2v", "resolution": args.resolution, "frames": args.frames, "fps": args.fps, "steps": args.steps,
                    "cfg": args.cfg, "seed": args.seed, "prompt": args.prompt},
        "timing": {"total_wall_s": round(time.monotonic() - t_start, 1), "generation_wall_s": res.get("wall_seconds"), **step_timing(res.get("updates", []))},
        "resources": stats, "output": probe, "error": res.get("error"),
        "health_before": res.get("health_before"), "health_after": res.get("health_after"),
        "progress_updates": res.get("updates", [])[-25:], "server_log_tail": server_log,
    }
    text = json.dumps(report, indent=1, default=str)
    assert "Bearer " not in text.replace("Bearer [redacted]", "")
    Path(args.report).parent.mkdir(parents=True, exist_ok=True)
    Path(args.report).write_text(text)
    emit({k: report[k] for k in ("verdict", "reasons", "claims_checked", "timing", "resources", "output", "error")})
    return 0 if verdict == "PASS" else 1


# ===================================================================================== CLI
def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("preflight")
    p.add_argument("--base", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--gpu", type=int, default=0); p.add_argument("--min-vram-gib", type=float, default=14.0)
    p.add_argument("--min-ram-gib", type=float, default=24.0); p.add_argument("--overhead-gib", type=float, default=8.0)
    p.add_argument("--repo", action="append", default=[], help="Git URL to check for reachability (repeatable)")
    p.add_argument("--offline", action="store_true", help="skip internet checks (for tests)")
    p.add_argument("--allow-no-gpu", action="store_true", help="downgrade 'no GPU' to a warning (for tests on CPU machines)")
    p.add_argument("--report")
    p = sub.add_parser("gpu-probe"); p.add_argument("--gpu", type=int, default=0)
    p = sub.add_parser("install-comfyui")
    p.add_argument("--base", required=True); p.add_argument("--commit", required=True)
    p.add_argument("--expect-version"); p.add_argument("--repo-url", default="https://github.com/comfyanonymous/ComfyUI.git")
    p = sub.add_parser("serve")
    p.add_argument("--base", required=True); p.add_argument("--port", type=int, default=8188)
    p.add_argument("--gpu", default="0"); p.add_argument("--wait", type=float, default=300)
    p.add_argument("extra", nargs="*", help="extra ComfyUI arguments (after --)")
    p = sub.add_parser("stop"); p.add_argument("--base", required=True)
    p = sub.add_parser("check-templates")
    p.add_argument("--url", default="http://127.0.0.1:8188")
    p.add_argument("--backend-dir", required=True); p.add_argument("--workflows-dir", required=True)
    p = sub.add_parser("smoke-t2v")
    p.add_argument("--base"); p.add_argument("--url", default="http://127.0.0.1:8188")
    p.add_argument("--backend-dir", required=True); p.add_argument("--workflows-dir", required=True)
    p.add_argument("--resolution", default="832x480"); p.add_argument("--frames", type=int, default=17)
    p.add_argument("--fps", type=int, default=24); p.add_argument("--steps", type=int, default=20)
    p.add_argument("--cfg", type=float, default=5.0); p.add_argument("--seed", type=int, default=424242)
    p.add_argument("--prompt", default=DEFAULT_PROMPT); p.add_argument("--gpu", type=int, default=0)
    p.add_argument("--timeout", type=float, default=2400); p.add_argument("--require-gpu", action="store_true")
    p.add_argument("--port", type=int, default=8188)
    p.add_argument("--fallback-args", default="", help='ComfyUI arguments for ONE retry after an out-of-memory failure, as one string; '
                   'use the = form: --fallback-args="--disable-dynamic-vram --disable-smart-memory"')
    p.add_argument("--out", required=True); p.add_argument("--report", required=True)
    p = sub.add_parser("verify-mp4")
    p.add_argument("file"); p.add_argument("--frames", type=int); p.add_argument("--size"); p.add_argument("--fps", type=float)
    a = ap.parse_args(argv)

    try:
        if a.cmd == "preflight":
            r = preflight(Path(a.base), Path(a.manifest), a.gpu, a.min_vram_gib, a.min_ram_gib, a.overhead_gib, a.offline,
                          a.allow_no_gpu, tuple(a.repo))
            for c in r["checks"]:
                print(f"[{c['status']:^4}] {c['name']}: {c['detail']}")
            print(f"\npreflight: {r['counts']['PASS']} PASS, {r['counts']['WARN']} WARN, {r['counts']['FAIL']} FAIL, {r['counts']['INFO']} INFO"
                  f" -> {'OK to continue' if r['ok'] else 'STOP: fix the FAIL items first'}")
            if a.report:
                Path(a.report).parent.mkdir(parents=True, exist_ok=True)
                Path(a.report).write_text(json.dumps(r, indent=1))
            return 0 if r["ok"] else 1
        elif a.cmd == "gpu-probe":
            emit(gpu_probe(a.gpu))
        elif a.cmd == "install-comfyui":
            emit(install_comfyui(Path(a.base), a.repo_url, a.commit, a.expect_version))
        elif a.cmd == "serve":
            emit(serve(Path(a.base), a.port, a.gpu, a.extra, a.wait))
        elif a.cmd == "stop":
            stop(Path(a.base))
        elif a.cmd == "check-templates":
            r = check_templates(a.url, Path(a.backend_dir), Path(a.workflows_dir))
            emit(r)
            return 0 if r["ok"] else 1
        elif a.cmd == "smoke-t2v":
            return smoke(a)
        elif a.cmd == "verify-mp4":
            size = tuple(int(x) for x in a.size.split("x")) if a.size else None
            r = probe_mp4(Path(a.file), a.frames, size, a.fps)
            emit(r)
            return 0 if not r["problems"] else 1
    except (RuntimeError, subprocess.SubprocessError, OSError) as e:
        print(f"FAILED: {redact(str(e))}", file=sys.stderr)
        return 2
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''')
print('wrote', Path(TOOLS, 'wan22_gpu_test.py'), Path(TOOLS, 'wan22_gpu_test.py').stat().st_size, 'bytes')

In [ ]:
# Embedded verbatim from workflows/comfyui/models.yaml, so this notebook is self-contained. A test keeps this copy identical.
Path(TOOLS, 'models.yaml').write_text(r'''# Exact model components for Wan 2.2 TI2V-5B in ComfyUI.
# Used by scripts/download_models.py and the Kaggle notebook (notebooks/kaggle_comfyui_wan22.ipynb).
# Weights are downloaded at runtime and are NEVER committed.
#
# License: Wan 2.2 is Apache-2.0 (Wan-AI/Wan2.2-TI2V-5B model card).
#
# --- How the integrity fields were established (2026-11-20) -----------------------------------
# Each repository's `revision` is the commit that `main` pointed to when this was recorded
# (GET https://huggingface.co/api/models/<repo>/revision/main -> sha). Downloads are pinned to it,
# so a later change to `main` can never alter what is downloaded.
#
# `sha256` is the SHA-256 of the FILE CONTENT. For these Git-LFS files Hugging Face stores it as the
# LFS object id, and it was read from two independent sources that agreed exactly:
#   1. GET https://huggingface.co/api/models/<repo>/paths-info/<revision>   -> lfs.oid
#   2. HEAD https://huggingface.co/<repo>/resolve/<revision>/<path> (redirects off) -> x-linked-etag
# `size_bytes` agreed across: this manifest, paths-info lfs.size and the x-linked-size header.
#
# Other identifiers Hugging Face returns for the same files are NOT SHA-256 and are not used:
#   * xetHash / x-xet-hash : the Xet storage chunk-tree hash, a different algorithm.
#   * oid (non-LFS)        : the Git blob id of the ~135-byte LFS pointer file, not of the weights.
#
# The downloader refuses to download any file whose `revision` or `sha256` is missing or malformed,
# and deletes any file whose size or SHA-256 does not match.
model: wan2.2-ti2v-5b
native_output:
  resolutions: ["1280x704", "704x1280"]   # TI2V-5B's 720p sizes
  fps: 24
  frame_rule: "4k+1"                      # e.g. 49, 73, 97, 121 (121 ~= 5 s)
files:
  - role: diffusion_model
    repo: Comfy-Org/Wan-2.2
    revision: 5df3a4a3ed0628002308022198249dbbbd045d09
    path: split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors
    comfyui_dir: models/diffusion_models
    size_bytes: 9999658848          # ~10.0 GB (no official fp8 build of the 5B exists)
    sha256: 456f901338bd9eadbded3828b819109a9b68e8a525ca5cf8d0049a69fcfeca1e

  - role: vae
    repo: Comfy-Org/Wan-2.2
    revision: 5df3a4a3ed0628002308022198249dbbbd045d09
    path: split_files/vae/wan2.2_vae.safetensors
    comfyui_dir: models/vae
    size_bytes: 1409400960          # ~1.4 GB. MUST be the Wan 2.2 VAE; the 2.1 VAE is incompatible.
    sha256: e40321bd36b9709991dae2530eb4ac303dd168276980d3e9bc4b6e2b75fed156

  - role: text_encoder
    repo: Comfy-Org/Wan-2.1
    revision: d6258323e18f14c60d37899da97ca555f3118259
    path: split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
    comfyui_dir: models/text_encoders
    size_bytes: 6735906897          # ~6.7 GB. fp8 chosen over fp16 (11.4 GB) to fit Kaggle RAM/VRAM.
    sha256: c3355d30191f1f066b26d93fba017ae9809dce6c627dda5f6a66eaa651204f68

total_bytes: 18144966705            # 16.9 GiB; the three sizes above sum to exactly this
total_download_gb: 18.1
# Explicitly NOT compatible with this pipeline (do not mix in; the downloader refuses these paths):
incompatible:
  - "Wan 2.2 14B T2V/I2V high_noise / low_noise models (different architecture, far more VRAM)"
  - "wan2.2_fun_* 5B/14B variants (different conditioning inputs)"
  - "Wan 2.1 VAE (wan_2.1_vae.safetensors)"
''')
print('wrote', Path(TOOLS, 'models.yaml'), Path(TOOLS, 'models.yaml').stat().st_size, 'bytes')

In [ ]:
# Embedded verbatim from scripts/backend_e2e_test.py, so this notebook is self-contained. A test keeps this copy identical.
Path(TOOLS, 'backend_e2e_test.py').write_text(r'''#!/usr/bin/env python3
"""VIDSGEN Phase 6B: a controlled backend-to-ComfyUI end-to-end test that runs ENTIRELY on one machine's loopback
(designed for a Kaggle GPU notebook). Nothing is exposed: no tunnel, no proxy, no public address.

    install-backend  Add the VIDSGEN backend's own dependencies to the existing ComfyUI venv, under the same protection
                     as the ComfyUI install: PyTorch, torchvision, torchaudio and NumPy are pinned to the host's copies and
                     the host's Python environment is checked to be unchanged.
    e2e              Start the real FastAPI backend on 127.0.0.1 (provider=comfyui, GENERATION_PROFILE=t4_verified, one job
                     at a time), then drive it ONLY through its public HTTP API: health, rejected out-of-envelope requests,
                     one real text-to-video job (832x480, 17 frames, 20 steps, 24 fps, CFG 5.0), progress polling, download,
                     Range request, and a full decode of the MP4. PASS requires every check; nothing is claimed from "the
                     server started" or "the job was queued".
    stop-backend     Stop the backend started by `e2e` (it also stops it itself when it finishes).

It reuses the helpers of scripts/wan22_gpu_test.py (which must sit next to this file) and does not change them.
"""

from __future__ import annotations

import argparse
import json
import os
import signal
import struct
import subprocess
import sys
import time
import tomllib
import zlib
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import wan22_gpu_test as gt  # noqa: E402  (the Phase 5B helpers)

# The one configuration verified on a Kaggle T4 in Phase 5B. The API defaults are NOT used: every value is sent explicitly.
PROVEN = {"mode": "t2v", "resolution": "832x480", "num_frames": 17, "steps": 20, "fps": 24, "guidance_scale": 5.0}
DEFAULT_PROMPT = gt.DEFAULT_PROMPT
LOOPBACK_HOSTS = ("127.0.0.1", "localhost", "::1", "[::1]")
ORDER = {"queued": 0, "processing": 1, "completed": 2, "failed": 2, "cancelled": 2}
_CHILDREN: dict[int, subprocess.Popen] = {}


# ===================================================================================== configuration
def loopback_endpoint(url: str) -> tuple[str, int]:
    """(host, port) of a plain-http loopback URL. Anything else is refused: this test must never leave the machine."""
    from urllib.parse import urlsplit

    p = urlsplit(url)
    if p.scheme != "http" or (p.hostname or "") not in {h.strip("[]") for h in LOOPBACK_HOSTS} or not p.port:
        raise ValueError(f"the ComfyUI URL must be http://127.0.0.1:<port> (loopback only), got {gt.redact(url)!r}")
    return p.hostname, p.port


def backend_requirements(backend_dir: Path) -> list[str]:
    """The backend's runtime dependencies, read from its pyproject.toml so they cannot drift."""
    deps = tomllib.loads((backend_dir / "pyproject.toml").read_text())["project"]["dependencies"]
    return [d.split("#")[0].strip() for d in deps]


def app_settings(base: Path, workflows_dir: Path, comfy_url: str, port: int) -> dict[str, str]:
    """The backend's settings for this test. Every value is non-secret and is recorded in the report."""
    loopback_endpoint(comfy_url)
    return {
        "APP_ENV": "production", "API_HOST": "127.0.0.1", "API_PORT": str(port), "LOG_LEVEL": "INFO",
        "CORS_ORIGINS": "",                                   # no browser origin is allowed: the API is driven by this script
        "INFERENCE_PROVIDER": "comfyui", "COMFYUI_BASE_URL": comfy_url, "COMFYUI_WORKFLOWS_DIR": str(workflows_dir),
        "COMFYUI_TIMEOUT": "60", "COMFYUI_CONNECTION_GRACE_SECONDS": "30", "INFERENCE_TIMEOUT_SECONDS": "1200",
        "GENERATION_PROFILE": "t4_verified", "INFERENCE_MAX_CONCURRENT_JOBS": "1",
        "STORAGE_LOCAL_ROOT": str(base / "storage"),
        # No COMFYUI_API_TOKEN: ComfyUI is on loopback, there is no proxy in this phase.
    }


def backend_env(base: Path, settings: dict[str, str], environ: dict[str, str] | None = None) -> dict[str, str]:
    """Environment for the backend process: ALLOW-LISTED variables only (no token or credential can reach it), caches
    inside the base directory, plus the settings above."""
    env = gt.server_env(base, None, environ)
    env.update(settings)
    return env


def png_bytes(width: int = 256, height: int = 256, rgb: tuple[int, int, int] = (30, 120, 200)) -> bytes:
    """A valid solid-colour PNG without any imaging library (used to probe the image-to-video rejection)."""
    def chunk(kind: bytes, data: bytes) -> bytes:
        return struct.pack(">I", len(data)) + kind + data + struct.pack(">I", zlib.crc32(kind + data) & 0xFFFFFFFF)

    raw = b"".join(b"\x00" + bytes(rgb) * width for _ in range(height))
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw)) + chunk(b"IEND", b""))


# ===================================================================================== install-backend
def install_backend(base: Path, backend_dir: Path, host_python: str = sys.executable) -> dict:
    """pip-install the backend's dependencies into <base>/venv, protecting the host's PyTorch/NumPy (see install_comfyui)."""
    venv, constraints = base / "venv", base / "constraints.txt"
    if not gt.venv_python(venv).exists() or not constraints.exists():
        raise RuntimeError(f"the ComfyUI venv and constraints file are missing in {base}: run install-comfyui first")
    for needed in ("pyproject.toml", "app/main.py", "app/jobs/envelope.py"):
        if not (backend_dir / needed).is_file():
            raise RuntimeError(f"{backend_dir / needed} is missing: the VIDSGEN checkout must include Phase 6A (the generation limits)")
    env = gt.clean_env(None, **gt.cache_env(base))
    inventory_before = gt.host_inventory(host_python, env, base)
    requirements = base / "requirements-backend.txt"
    requirements.write_text("\n".join(backend_requirements(backend_dir)) + "\n")
    pip = gt.pip_info(str(gt.venv_python(venv)), venv, env)
    cmd = [str(gt.venv_python(venv)), "-m", "pip", "install", "--no-cache-dir", "--disable-pip-version-check", "-q",
           "-c", str(constraints), "-r", str(requirements)]
    t0 = time.monotonic()
    p = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=3600)
    if p.returncode != 0:
        raise RuntimeError("pip install failed (PyTorch and NumPy were protected by the constraints file):\n"
                           + gt.redact(p.stdout + p.stderr)[-1500:])
    host_seen, seen = gt.import_probe(host_python, env), gt.import_probe(str(gt.venv_python(venv)), env)
    problems = gt.protected_packages_replaced(host_seen, seen, venv)
    if problems:
        raise RuntimeError("the host's PyTorch/NumPy stack was replaced: " + "; ".join(problems))
    changes = gt.inventory_changes(inventory_before, gt.host_inventory(host_python, env, base))
    if changes:
        raise RuntimeError("the host's Python environment was modified by the install (it must stay untouched): "
                           + "; ".join(changes[:12]))
    importable = backend_importable(str(gt.venv_python(venv)), backend_dir, base)
    if importable:
        raise RuntimeError("the backend cannot be imported from the venv after the install:\n" + importable)
    chk = subprocess.run([str(gt.venv_python(venv)), "-m", "pip", "check"], capture_output=True, text=True, env=env, timeout=300)
    return {"venv": str(venv), "pip": pip, "requirements": backend_requirements(backend_dir), "pip_seconds": round(time.monotonic() - t0),
            "protected_packages": {n: {"version": v[0], "identical_to_host": True} for n, v in seen["packages"].items()
                                   if host_seen["packages"].get(n)},
            "host_environment_unchanged": True, "backend_importable": True,
            "pip_check": (chk.stdout + chk.stderr).strip()[-600:] or "No broken requirements found."}


def backend_importable(python: str, backend_dir: Path, base: Path) -> str:
    """'' if `app.main` imports with this interpreter, else the (redacted) reason."""
    env = gt.clean_env(None, **gt.cache_env(base))
    env.update(INFERENCE_PROVIDER="mock", STORAGE_LOCAL_ROOT=str(base / "storage-import-check"))
    p = subprocess.run([python, "-c", "import app.main"], cwd=backend_dir, capture_output=True, text=True, env=env, timeout=120)
    return "" if p.returncode == 0 else gt.redact((p.stderr or p.stdout)[-800:])


# ===================================================================================== backend process
def start_backend(base: Path, backend_dir: Path, python: str, settings: dict[str, str], wait: float = 90) -> dict:
    port = int(settings["API_PORT"])
    (base / "logs").mkdir(parents=True, exist_ok=True)
    stop_backend(base, quiet=True)
    pidfile, logfile = base / "logs" / "backend.pid", base / "logs" / "backend.log"
    cmd = [python, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", str(port), "--log-level", "info"]
    with open(logfile, "wb") as log:
        proc = subprocess.Popen(cmd, cwd=backend_dir, env=backend_env(base, settings), stdout=log, stderr=subprocess.STDOUT,
                                start_new_session=True)
    pidfile.write_text(str(proc.pid))
    _CHILDREN[proc.pid] = proc
    deadline = time.monotonic() + wait
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            stop_backend(base, quiet=True)
            raise RuntimeError(f"the backend exited with code {proc.returncode}:\n" + gt.redact(gt.tail(logfile, 30)))
        try:
            if gt.http_get(f"http://127.0.0.1:{port}/api/health")[0] == 200:
                addrs = gt.listening_addresses(port)
                return {"pid": proc.pid, "port": port, "listening_on": addrs,
                        "loopback_only": bool(addrs) and all(gt.is_loopback(a) for a in addrs), "log": str(logfile)}
        except OSError:
            pass
        time.sleep(0.5)
    stop_backend(base, quiet=True)
    raise RuntimeError(f"the backend did not become ready within {wait:.0f}s:\n" + gt.redact(gt.tail(logfile, 30)))


def stop_backend(base: Path, quiet: bool = False) -> bool:
    pidfile = base / "logs" / "backend.pid"
    if not pidfile.exists():
        return False
    try:
        pid = int(pidfile.read_text())
    except ValueError:
        pidfile.unlink(missing_ok=True)
        return False
    proc = _CHILDREN.pop(pid, None)
    try:
        group = os.getpgid(pid)
        os.killpg(group, signal.SIGTERM)
        for _ in range(60):
            time.sleep(0.25)
            if (proc.poll() is not None) if proc is not None else not gt._alive(pid):
                break
        else:
            os.killpg(group, signal.SIGKILL)
    except OSError:
        pass
    if proc is not None:
        try:
            proc.wait(timeout=10)
        except subprocess.SubprocessError:
            pass
    pidfile.unlink(missing_ok=True)
    if not quiet:
        print("backend stopped")
    return True


# ===================================================================================== the test
class Verdict:
    """Collects named claims. PASS only if every claim that was evaluated is True."""

    def __init__(self):
        self.claims: dict[str, bool | None] = {}
        self.reasons: list[str] = []

    def claim(self, name: str, ok: bool, why: str = "") -> bool:
        self.claims[name] = bool(ok)
        if not ok:
            self.reasons.append(f"{name}: {why}" if why else name)
        return bool(ok)

    def skip(self, name: str) -> None:
        self.claims[name] = None

    @property
    def verdict(self) -> str:
        return "PASS" if self.claims and all(v is not False for v in self.claims.values()) and not self.reasons else "FAIL"


def _check_summary(health: dict) -> dict[str, str]:
    return {c["name"]: c["status"] for c in health.get("provider", {}).get("checks", [])}


def poll_job(api, job_id: str, timeout: float, interval: float, sampler=None) -> dict:
    """Poll GET /api/jobs/{id} until it is terminal. Returns the timeline and the last job body."""
    t0 = time.monotonic()
    timeline: list[dict] = []
    job: dict = {}
    last_beat = 0.0
    while time.monotonic() - t0 < timeout:
        r = api.get(f"/api/jobs/{job_id}")
        r.raise_for_status()
        job = r.json()
        now = round(time.monotonic() - t0, 2)
        if not timeline or (timeline[-1]["status"], timeline[-1]["progress"]) != (job["status"], job["progress"]):
            timeline.append({"t": now, "status": job["status"], "progress": job["progress"]})
        if now - last_beat >= 10 or job["status"] in ("completed", "failed", "cancelled"):
            last_beat = now
            vram = f" | GPU memory {sampler.vram_used_mib[-1]} MiB" if sampler is not None and sampler.vram_used_mib else ""
            print(f"  [{now:6.0f}s] {job['status']:<10} {job['progress']:5.1f}%{vram}", file=sys.stderr, flush=True)
        if job["status"] in ("completed", "failed", "cancelled"):
            return {"timeline": timeline, "job": job, "timed_out": False, "seconds": now}
        time.sleep(interval)
    return {"timeline": timeline, "job": job, "timed_out": True, "seconds": round(time.monotonic() - t0, 2)}


def run_e2e(args) -> int:
    try:
        import httpx
    except ImportError as e:
        print(f"FAILED: {e}. Run this command with the ComfyUI venv's python (<base>/venv/bin/python).", file=sys.stderr)
        return 2
    base, backend_dir = Path(args.base), Path(args.backend_dir)
    python = args.python or str(gt.venv_python(base / "venv"))
    comfy_host, comfy_port = loopback_endpoint(args.comfyui_url)
    settings = app_settings(base, Path(args.workflows_dir), args.comfyui_url, args.backend_port)
    reason = backend_importable(python, backend_dir, base)
    if reason:
        raise RuntimeError("the backend cannot be imported with " + python + " (run install-backend first):\n" + reason)

    v = Verdict()
    report: dict = {"settings_applied": settings, "request": {**PROVEN, "seed": args.seed, "prompt": args.prompt}}
    api = None
    sampler = None
    started = False
    t_start = time.monotonic()
    comfy_pidfile = base / "logs" / "comfyui.pid"
    comfy_pid = int(comfy_pidfile.read_text()) if comfy_pidfile.exists() and comfy_pidfile.read_text().strip().isdigit() else None
    try:
        # 1. Services and their bindings --------------------------------------------------------------------------
        backend = start_backend(base, backend_dir, python, settings)
        report["backend"] = backend
        started = True
        v.claim("backend_started", True)
        v.claim("backend_loopback_only", backend["loopback_only"], f"listening on {backend['listening_on']}")
        comfy_addrs = gt.listening_addresses(comfy_port)
        report["comfyui_listening_on"] = comfy_addrs
        v.claim("comfyui_loopback_only", bool(comfy_addrs) and all(gt.is_loopback(a) for a in comfy_addrs), f"listening on {comfy_addrs}")
        api = httpx.Client(base_url=f"http://127.0.0.1:{args.backend_port}", timeout=60)

        # 2. Health -----------------------------------------------------------------------------------------------
        health = api.get("/api/health").json()
        report["health_before"] = health
        v.claim("provider_is_comfyui", health["provider"]["name"] == "comfyui", f"provider is {health['provider']['name']}")
        v.claim("provider_ready", health["provider"]["ok"], health["provider"]["detail"])

        # 3. The envelope is active in this deployment (cheap: nothing reaches the GPU) ----------------------------
        jobs_before = api.get("/api/jobs").json()["total"]
        probes = {
            "resolution": {**_form(), "resolution": "1280x704"},
            "num_frames": {**_form(), "num_frames": "49"},
            "steps": {**_form(), "steps": "30"},
            "mode": {**_form(), "mode": "i2v"},
        }
        rejected: dict[str, dict] = {}
        for field, data in probes.items():
            files = {"image": ("probe.png", png_bytes(), "image/png")} if field == "mode" else None
            r = api.post("/api/jobs", data=data, files=files)
            locs = [e["loc"][-1] for e in r.json().get("detail", [])] if r.status_code == 422 and isinstance(r.json().get("detail"), list) else []
            rejected[field] = {"status": r.status_code, "fields": locs}
        report["envelope_probes"] = rejected
        no_new_jobs = api.get("/api/jobs").json()["total"] == jobs_before
        v.claim("envelope_enforced", all(p["status"] == 422 and f in p["fields"] for f, p in rejected.items()) and no_new_jobs,
                f"probes: {rejected}; no new jobs: {no_new_jobs}")

        # 4. The real job, with the proven values ---------------------------------------------------------------------
        sampler = gt.Sampler(gpu=args.gpu, pid=comfy_pid)
        sampler.start()
        sampler.sample_once()
        sampler.baseline_vram_mib = sampler.vram_used_mib[-1] if sampler.vram_used_mib else None
        body = {**_form(), "prompt": args.prompt, "seed": str(args.seed)}
        r = api.post("/api/jobs", data=body)
        report["submit"] = {"status_code": r.status_code}
        if not v.claim("job_accepted", r.status_code == 202, f"POST /api/jobs returned {r.status_code}: {gt.redact(r.text[:300])}"):
            raise _Stop
        job_id = r.json()["id"]
        report["job_id"] = job_id
        polled = poll_job(api, job_id, args.timeout, args.poll, sampler)
        job, timeline = polled["job"], polled["timeline"]
        report["timeline"] = timeline[:40] + ([{"...": f"{len(timeline) - 60} more"}] + timeline[-20:] if len(timeline) > 60 else timeline[40:])
        report["job_seconds"] = polled["seconds"]
        statuses = [t["status"] for t in timeline]
        v.claim("status_flow_valid", all(ORDER[a] <= ORDER[b] for a, b in zip(statuses, statuses[1:])) and "processing" in statuses,
                f"statuses seen: {statuses}")
        progress = [t["progress"] for t in timeline]
        v.claim("progress_monotonic", all(a <= b for a, b in zip(progress, progress[1:])), f"progress seen: {progress}")
        v.claim("progress_intermediate_values", any(0 < t["progress"] < 100 for t in timeline if t["status"] == "processing"),
                "polling never saw a progress value between 0 and 100 while processing")
        if polled["timed_out"]:
            api.post(f"/api/jobs/{job_id}/cancel")
            v.claim("job_completed", False, f"no result within {args.timeout:.0f}s (cancelled)")
            raise _Stop
        report["job"] = job
        if not v.claim("job_completed", job["status"] == "completed",
                       f"job ended as {job['status']}: {json.dumps(job.get('error'))}"):
            raise _Stop

        # 5. The job record --------------------------------------------------------------------------------------------
        p = job["params"]
        v.claim("job_record_matches_request", job["provider"] == "comfyui" and job["error"] is None and
                (p["resolution"], p["num_frames"], p["steps"], p["fps"], p["guidance_scale"], p["seed"]) ==
                ("832x480", 17, 20, 24, 5.0, args.seed), f"provider={job['provider']} params={p}")
        v.claim("job_in_history", any(j["id"] == job_id for j in api.get("/api/jobs").json()["items"]), "not listed in GET /api/jobs")

        # 6. Video retrieval -------------------------------------------------------------------------------------------------
        out = Path(args.out)
        out.parent.mkdir(parents=True, exist_ok=True)
        download_url = job["result"]["download_url"]
        with api.stream("GET", download_url) as resp:
            ok_headers = resp.status_code == 200 and resp.headers.get("content-type", "").startswith("video/mp4") and \
                resp.headers.get("content-disposition", "").startswith("attachment")
            with open(out, "wb") as f:
                for chunk in resp.iter_bytes(1024 * 1024):
                    f.write(chunk)
        v.claim("download_ok", ok_headers and out.stat().st_size == job["result"]["size_bytes"] > 0,
                f"status/headers/size mismatch (file {out.stat().st_size if out.exists() else 0} bytes, job says {job['result']['size_bytes']})")
        data = out.read_bytes()
        rr = api.get(download_url, headers={"Range": "bytes=0-1023"})
        v.claim("range_request_ok", rr.status_code == 206 and rr.content == data[:1024] and
                rr.headers.get("content-range", "").startswith("bytes 0-1023/"), f"HTTP {rr.status_code}")
        pv = api.get(job["result"]["preview_url"], headers={"Range": "bytes=0-11"})
        v.claim("preview_inline_ok", pv.status_code in (200, 206) and pv.headers.get("content-disposition", "").startswith("inline") and
                pv.content[4:8] == b"ftyp", f"HTTP {pv.status_code}")

        # 7. The MP4 itself (fully decoded) -------------------------------------------------------------------------------
        probe = gt.probe_mp4(out, frames=PROVEN["num_frames"], size=(832, 480), fps=float(PROVEN["fps"]))
        report["output"] = probe
        v.claim("mp4_decodes_and_matches", not probe["problems"], "; ".join(probe["problems"]))

        # 8. Health after a real generation --------------------------------------------------------------------------------
        health2 = api.get("/api/health").json()
        report["health_after"] = health2
        v.claim("inference_verified_by_generation", _check_summary(health2).get("inference") == "ok",
                f"checks: {_check_summary(health2)}")

        # 9. Optional: cancel a second job (real interrupt) ---------------------------------------------------------------------
        if args.with_cancel:
            cancel = cancel_test(api, body, args.poll)
            report["cancel_test"] = cancel
            v.claim("cancel_works", cancel["ok"], cancel["why"])
        else:
            v.skip("cancel_works")
    except _Stop:
        pass
    except Exception as e:  # noqa: BLE001  (reported, never a traceback with secrets)
        v.claim("test_ran_to_completion", False, f"{type(e).__name__}: {gt.redact(str(e))[:300]}")
    finally:
        stats = sampler.finish() if sampler is not None else {}
        if api is not None:
            api.close()
        stopped = stop_backend(base, quiet=True)
        if started:
            v.claim("backend_stopped", stopped and not gt.listening_addresses(args.backend_port), "the backend is still listening")
        if args.require_gpu and stats and stats.get("gpu_sampling") == "unavailable":
            v.claim("gpu_measured", False, "GPU memory could not be measured (nvidia-smi unavailable)")

    verdict = v.verdict
    report.update({
        "verdict": verdict, "reasons": v.reasons, "claims": v.claims,
        "resources": stats,
        "timing": {"total_wall_s": round(time.monotonic() - t_start, 1), "job_wall_s": report.get("job_seconds")},
        "backend_log_tail": gt.redact(gt.tail(base / "logs" / "backend.log", 60)) if verdict == "FAIL" else None,
    })
    text = json.dumps(report, indent=1, default=str)
    assert "Bearer " not in text.replace("Bearer [redacted]", ""), "a token-like string reached the report"
    Path(args.report).parent.mkdir(parents=True, exist_ok=True)
    Path(args.report).write_text(text)
    gt.emit({k: report[k] for k in ("verdict", "reasons", "claims", "timing", "resources") if k in report} |
            ({"output": report["output"]} if "output" in report else {}))
    return 0 if verdict == "PASS" else 1


class _Stop(Exception):
    """Ends the test early; the reasons are already recorded."""


def _form() -> dict[str, str]:
    """The explicit, proven request fields (form-encoded)."""
    return {"mode": PROVEN["mode"], "prompt": DEFAULT_PROMPT, "resolution": PROVEN["resolution"],
            "num_frames": str(PROVEN["num_frames"]), "steps": str(PROVEN["steps"]), "fps": str(PROVEN["fps"]),
            "guidance_scale": str(PROVEN["guidance_scale"])}


def cancel_test(api, body: dict, interval: float, wait_processing: float = 180) -> dict:
    """Submit another job, cancel it while it is being processed, and check it stays cancelled."""
    r = api.post("/api/jobs", data=body)
    if r.status_code != 202:
        return {"ok": False, "why": f"submit returned {r.status_code}"}
    job_id = r.json()["id"]
    t0 = time.monotonic()
    while time.monotonic() - t0 < wait_processing:
        status = api.get(f"/api/jobs/{job_id}").json()["status"]
        if status == "processing":
            break
        if status in ("completed", "failed", "cancelled"):
            return {"ok": False, "why": f"job ended as {status} before it could be cancelled", "job_id": job_id}
        time.sleep(interval)
    c = api.post(f"/api/jobs/{job_id}/cancel")
    time.sleep(max(interval, 2.0))
    final = api.get(f"/api/jobs/{job_id}").json()
    ok = c.status_code == 200 and c.json()["status"] == "cancelled" and final["status"] == "cancelled" and final["result"] is None
    return {"ok": ok, "why": "" if ok else f"cancel returned {c.status_code}; final status {final['status']}", "job_id": job_id}


# ===================================================================================== CLI
def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("install-backend")
    p.add_argument("--base", required=True); p.add_argument("--backend-dir", required=True)
    p = sub.add_parser("e2e")
    p.add_argument("--base", required=True); p.add_argument("--backend-dir", required=True); p.add_argument("--workflows-dir", required=True)
    p.add_argument("--python", help="interpreter that runs the backend (default: <base>/venv/bin/python)")
    p.add_argument("--comfyui-url", default="http://127.0.0.1:8188"); p.add_argument("--backend-port", type=int, default=8000)
    p.add_argument("--gpu", type=int, default=0); p.add_argument("--timeout", type=float, default=2400)
    p.add_argument("--poll", type=float, default=1.0); p.add_argument("--seed", type=int, default=424242)
    p.add_argument("--prompt", default=DEFAULT_PROMPT); p.add_argument("--require-gpu", action="store_true")
    p.add_argument("--with-cancel", action="store_true", help="also cancel a second job while it is being processed")
    p.add_argument("--out", required=True); p.add_argument("--report", required=True)
    p = sub.add_parser("stop-backend"); p.add_argument("--base", required=True)
    a = ap.parse_args(argv)
    try:
        if a.cmd == "install-backend":
            gt.emit(install_backend(Path(a.base), Path(a.backend_dir)))
        elif a.cmd == "e2e":
            return run_e2e(a)
        elif a.cmd == "stop-backend":
            stop_backend(Path(a.base))
    except (RuntimeError, ValueError, subprocess.SubprocessError, OSError) as e:
        print(f"FAILED: {gt.redact(str(e))}", file=sys.stderr)
        return 2
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''')
print('wrote', Path(TOOLS, 'backend_e2e_test.py'), Path(TOOLS, 'backend_e2e_test.py').stat().st_size, 'bytes')

In [ ]:
# --- 3. Stage 1: preflight (nothing is installed or downloaded yet) ----------------------------------------
# Gates: Python >= 3.11, a non-persistent install location with enough free disk, GPU 0 + VRAM + CUDA/PyTorch
# compatibility, RAM, internet, and the authoritative Hugging Face metadata (pinned revision, size, SHA-256).
code, out = run([sys.executable, f"{TOOLS}/wan22_gpu_test.py", "preflight", "--base", BASE, "--manifest", f"{TOOLS}/models.yaml",
                 "--gpu", GPU, "--repo", VIDSGEN_REPO, "--repo", COMFYUI_REPO, "--report", f"{OUT}/vidsgen_preflight_5b.json"],
                env=hf_env(), check=False)
STAGES["preflight"] = {"ok": code == 0, **json.loads(Path(f"{OUT}/vidsgen_preflight_5b.json").read_text())}
if code != 0:
    stage_failed("preflight", "one or more checks FAILED (see the [FAIL] lines above). Nothing was installed or downloaded.")

In [ ]:
# --- 4. Stage 2a: the pinned VIDSGEN commit (existing workflows, ComfyUIProvider, validators) ----------------
sys.path.insert(0, TOOLS)
import wan22_gpu_test as gt

head = gt.checkout_commit(VIDSGEN_REPO, VIDSGEN_COMMIT, Path(BASE, "vidsgen"))
needed = ["workflows/comfyui/wan22_ti2v_5b_t2v.json", "workflows/comfyui/wan22_ti2v_5b_i2v.json", "workflows/comfyui/wan22_ti2v_5b.mapping.json",
          "backend/app/providers/comfyui.py", "backend/app/workflows/templates.py", "backend/tests/comfyui_support.py"]
missing = [f for f in needed if not Path(BASE, "vidsgen", f).is_file()]
if missing:
    stage_failed("vidsgen_checkout", f"files missing from the pinned commit: {missing}")
STAGES["vidsgen_checkout"] = {"ok": True, "commit": head}
print("VIDSGEN checked out at", head)

In [ ]:
# --- 5. Stage 2b: ComfyUI at the pinned commit, in a venv that REUSES Kaggle's PyTorch --------------------
# pip runs with a constraints file pinning torch/torchvision/torchaudio/numpy to what is already installed, and the
# result is verified: the venv must import the host's identical PyTorch file. Takes a few minutes.
code, out = run([sys.executable, f"{TOOLS}/wan22_gpu_test.py", "install-comfyui", "--base", BASE,
                 "--commit", COMFYUI_COMMIT, "--expect-version", COMFYUI_VERSION], env=hf_env(), check=False)
if code != 0:
    stage_failed("comfyui_install", "installation failed (see the error above); the host's PyTorch was not modified.")
STAGES["comfyui_install"] = {"ok": True, **last_json(out)}

In [ ]:
# --- 6. Stage 3: download + verify the three model files (exactly 16.9 GiB) -----------------------------------
# Only the files in the manifest, from their pinned revisions. Size AND SHA-256 are checked; a mismatch deletes the
# file and stops. An interrupted download resumes (HTTP Range) and is trusted only after the whole-file SHA-256 matches.
if not DOWNLOAD_MODELS:
    STAGES["models"] = {"ok": None, "skipped": "DOWNLOAD_MODELS is False"}
    print("DOWNLOAD_MODELS is False: skipping the download. Later stages need the model files.")
else:
    code, _ = run([sys.executable, f"{TOOLS}/download_models.py", "--manifest", f"{TOOLS}/models.yaml", "--comfyui-dir", DATA, "--execute"],
                  env=hf_env(), check=False)
    if code != 0:
        why = {2: "refused before downloading (manifest or disk space)", 3: "network problem; the partial download is kept: just re-run this cell",
               4: "VERIFICATION FAILED: the file was deleted"}.get(code, f"exit code {code}")
        stage_failed("models", why)
    print("\n--- independent re-verification of the files on disk (size + SHA-256, no network) ---")
    code, out = run([sys.executable, f"{TOOLS}/download_models.py", "--manifest", f"{TOOLS}/models.yaml", "--comfyui-dir", DATA, "--verify-only"],
                    check=False)
    if code != 0:
        stage_failed("models", "re-verification of the downloaded files failed")
    STAGES["models"] = {"ok": True, "verified": "exact size + SHA-256 (download-time and again from disk)", "files": 3}

In [ ]:
# --- 7. Stage 4: start ComfyUI (127.0.0.1 only, GPU 0 only, credential-free environment) ---------------------
if STAGES["models"].get("ok") is not True:
    stage_failed("server", "the model files are not verified; not starting ComfyUI")
code, out = run([sys.executable, f"{TOOLS}/wan22_gpu_test.py", "serve", "--base", BASE, "--port", str(PORT), "--gpu", GPU, "--wait", "300"],
                check=False)
if code != 0:
    stage_failed("server", "ComfyUI did not start cleanly (see the error and log tail above)")
server = last_json(out)
STAGES["server"] = {"ok": True, **server}
print("\nloopback only:", server["loopback_only"], "| listening on:", server["listening_on"], "| CUDA_VISIBLE_DEVICES:", server["cuda_visible_devices"])
print("\n--- ComfyUI start-up log (first 30 lines; secrets redacted) ---")
print(gt.redact("\n".join(Path(server["log"]).read_text(errors="replace").splitlines()[:30])))

In [ ]:
# --- 8. Stage 5: model discovery + both workflow templates against the LIVE server ---------------------------
import urllib.request


def api(path):
    with urllib.request.urlopen(URL + path, timeout=30) as r:
        return json.loads(r.read())


stats = api("/system_stats")
dev = stats["devices"][0]
print(f"ComfyUI {stats['system']['comfyui_version']} | device: {dev['name']} | VRAM {dev['vram_total'] / 2**30:.1f} GiB total, "
      f"{dev['vram_free'] / 2**30:.1f} GiB free | PyTorch {stats['system']['pytorch_version']}")
manifest_files = {"diffusion_models": ("wan2.2_ti2v_5B_fp16.safetensors", 9999658848), "text_encoders": ("umt5_xxl_fp8_e4m3fn_scaled.safetensors", 6735906897),
                  "vae": ("wan2.2_vae.safetensors", 1409400960)}
discovery = {}
for folder, (name, size) in manifest_files.items():
    listed = {m["name"]: m["size"] for m in api(f"/experiment/models/{folder}")}
    discovery[folder] = {"discovered": name in listed, "size_matches": listed.get(name) == size}
    print(f"  models/{folder}: {name} -> discovered={discovery[folder]['discovered']}, exact size={discovery[folder]['size_matches']}")
if not all(d["discovered"] and d["size_matches"] for d in discovery.values()):
    stage_failed("verify_server", "ComfyUI does not see all three model files with their exact sizes")

code, out = run([VENV_PY, f"{TOOLS}/wan22_gpu_test.py", "check-templates", "--url", URL, "--backend-dir", f"{BASE}/vidsgen/backend",
                 "--workflows-dir", f"{BASE}/vidsgen/workflows/comfyui"], check=False)
templates = last_json(out)
STAGES["verify_server"] = {"ok": code == 0, "discovery": discovery, "templates": templates,
                           "device": dev["name"], "vram_total_bytes": dev["vram_total"]}
if code != 0:
    stage_failed("verify_server", f"workflow templates do not match this server: missing nodes {templates['node_classes_missing']}, "
                                  f"problems {templates['templates']}")

In [ ]:
# --- 8. Stage 6: the backend's dependencies, into the SAME venv, with PyTorch/NumPy protected --------------------
# Runs under the HOST python so the host's PyTorch/NumPy can be compared with the venv's. Afterwards the venv must import the
# host's identical torch, torchvision, torchaudio and numpy, the host's installed packages must be unchanged, and `import app.main` must work.
code, out = run([sys.executable, f"{TOOLS}/backend_e2e_test.py", "install-backend", "--base", BASE, "--backend-dir", f"{BASE}/vidsgen/backend"],
                env=hf_env(), check=False)
if code != 0:
    stage_failed("backend_install", "installing the backend failed (see the error above); the host's PyTorch and NumPy were not modified.")
STAGES["backend_install"] = {"ok": True, **last_json(out)}
print("\nbackend installed | host environment unchanged:", STAGES["backend_install"]["host_environment_unchanged"],
      "| protected:", sorted(STAGES["backend_install"]["protected_packages"]))

In [ ]:
# --- 9. Stage 7: the end-to-end test (backend <-> ComfyUI over 127.0.0.1) -------------------------------------
# The backend runs as a separate process (provider=comfyui, GENERATION_PROFILE=t4_verified, one job at a time, loopback only) and is driven
# ONLY through its HTTP API. The first generation loads the models: watch the heartbeat lines. Nothing here is a mock.
if not RUN_E2E:
    STAGES["backend_e2e"] = {"ok": None, "skipped": "RUN_E2E is False"}
    print("RUN_E2E is False: skipping.")
else:
    report_path, video_path = f"{OUT}/vidsgen_phase6b_e2e_report.json", f"{OUT}/vidsgen_phase6b_e2e.mp4"
    cmd = [VENV_PY, f"{TOOLS}/backend_e2e_test.py", "e2e", "--base", BASE, "--backend-dir", f"{BASE}/vidsgen/backend",
           "--workflows-dir", f"{BASE}/vidsgen/workflows/comfyui", "--comfyui-url", URL, "--backend-port", str(BACKEND_PORT),
           "--gpu", GPU, "--timeout", str(E2E_TIMEOUT_S), "--require-gpu", "--out", video_path, "--report", report_path]
    if RUN_CANCEL_TEST:
        cmd.append("--with-cancel")
    code, _ = run(cmd, check=False)
    e2e = json.loads(Path(report_path).read_text())
    STAGES["backend_e2e"] = {"ok": code == 0, "report": report_path, "video": video_path if code == 0 else None,
                             **{k: e2e.get(k) for k in ("verdict", "reasons", "claims", "timing", "resources", "envelope_probes", "output")}}
    print("\n" + "=" * 78)
    print(f"BACKEND END-TO-END VERDICT: {e2e['verdict']}")
    print("=" * 78)
    for name, ok in e2e["claims"].items():
        print(f"  {'PASS' if ok else ('skip' if ok is None else 'FAIL')}  {name}")
    res, tim = e2e.get("resources", {}), e2e.get("timing", {})
    print(f"\npeak GPU memory: {res.get('vram_peak_used_mib')} MiB of {res.get('vram_total_mib')} MiB | peak RAM {res.get('ram_peak_used_gib')} GiB "
          f"of {res.get('ram_total_gib')} GiB | job {tim.get('job_wall_s')} s, total {tim.get('total_wall_s')} s")
    if code == 0:
        o = e2e["output"]
        print(f"output: {video_path} ({o['bytes']:,} bytes; {o['container']}, {o['codec']}, {o['width']}x{o['height']}, "
              f"{o['decoded_frames']} frames, {o['fps']} fps)")
        from IPython.display import Video, display
        display(Video(video_path, embed=True, width=480))
    else:
        print("reasons:", e2e["reasons"])
        stage_failed("backend_e2e", "; ".join(e2e["reasons"]))

In [ ]:
# --- 10. Final report (contains no secrets) ----------------------------------------------------------------
report = {"generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "stages": STAGES,
          "pinned": {"vidsgen_commit": VIDSGEN_COMMIT, "comfyui_commit": COMFYUI_COMMIT, "comfyui_version": COMFYUI_VERSION},
          "settings": {"gpu": GPU, "backend_port": BACKEND_PORT, "profile": "t4_verified", "max_concurrent_jobs": 1,
                       "request": dict(resolution="832x480", frames=17, steps=20, fps=24, cfg=5.0), "cancel_test": RUN_CANCEL_TEST, "base": BASE}}
text = json.dumps(report, indent=1, default=str)
assert "Bearer " not in text and not re.search(r"hf_[A-Za-z0-9]{20,}", text), "token-like text found; report not written"
Path(f"{OUT}/vidsgen_phase6b_report.json").write_text(text)

print("Stage results (True = passed, None = skipped by your settings):")
for name, s in STAGES.items():
    print(f"  {name:<18} {s.get('ok')}")
passed = STAGES.get("backend_e2e", {}).get("ok") is True
print("\n" + ("THE BACKEND GENERATED A REAL VIDEO THROUGH COMFYUI ON THIS GPU, AND EVERY CHECK PASSED (localhost only)." if passed else
              "The backend-to-GPU end-to-end test did NOT pass in this run: do not treat the integration as working."))
print("\nNot validated by this notebook: image-to-video, a frontend in front of this backend, the proxy/tunnel, any production deployment.")
print(f"\nSend back: {OUT}/vidsgen_phase6b_report.json and {OUT}/vidsgen_phase6b_e2e_report.json (no secrets).")
if STOP_SERVER_AT_END:
    run([sys.executable, f"{TOOLS}/wan22_gpu_test.py", "stop", "--base", BASE], check=False)
    run([sys.executable, f"{TOOLS}/backend_e2e_test.py", "stop-backend", "--base", BASE], check=False)
else:
    print(f"ComfyUI is still running on 127.0.0.1:{PORT} (loopback only). No tunnel was started.")

## What to send back
* `/kaggle/working/vidsgen_phase6b_report.json` and `/kaggle/working/vidsgen_phase6b_e2e_report.json`: no secrets, safe to share.
* If a stage stopped the run, the `[FAIL]` / `STOP` lines above it.

## Stop everything
The last cell stops ComfyUI and the backend (`STOP_SERVER_AT_END`). Ending the session also stops them.

## Not in this notebook (separate approval)
The token-protected proxy, any tunnel, image-to-video on the GPU, and the frontend.